In [ ]:
import os, json, csv, io, re, time, glob, hashlib
from datetime import date
from google.colab import drive, userdata
from google import genai

# 1. Connect Drive
drive.mount('/content/drive')

# 2. Install Gemini library
import subprocess
subprocess.run(["pip", "install", "-q", "-U", "google-genai"])

# 3. Paths
PROJECT_ROOT = "/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT"
SYNTHETIC_ROOT = os.path.join(PROJECT_ROOT, "03_synthetic_data")

# 4. Connect to Gemini
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"

# 5. Helper to save CSV files
def save_csv(rows, filepath, fieldnames):
    with open(filepath, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

# 6. General prompt + functions
GENERAL_PROMPT_TEMPLATE = """
You generate short, natural Saudi-style Arabic texts for a research corpus.

Goal: Create unlabeled general Saudi Arabic text for continued pre-training. The text will be used only to expose a language model to Saudi-style wording before a separate banking intent-classification task.

Output format: Return CSV rows only, with these columns:
id,text,pool,topic,genre,length_bucket,generator,prompt_version,batch_id,created_at,audit_status
Use `general` for pool, `v1.1` for prompt_version, and `pending` for audit_status.

Content requirements:
- Write natural Saudi-style Arabic, not formal MSA and not deliberately broken Arabic.
- Produce only one self-contained text per row.
- Use the requested topic, genre, and length bucket.
- Use everyday, non-financial topics only.
- Do not include personal names, phone numbers, account numbers, addresses, IDs, or private information.
- Avoid repeated templates, repeated openings, and near-duplicate texts.
- Use globally unique IDs with the batch prefix, for example: general_pilot_01_001.
- Avoid ambiguous standalone words that may be banking-related, such as "branch", unless the non-financial context is explicit in the same text.

Strict exclusions: Do NOT mention or imply banks, banking applications, cards, transfers, balances, payments, invoices, money, currencies, loans, accounts, ATMs, or any banking intent from ArBanking77.

Genre guidance:
- question: a natural question someone may ask in daily life.
- statement: a short everyday statement or observation.
- comment: a casual comment or review about a non-financial everyday topic.

Length guidance:
- short: 5-9 Arabic words.
- medium: 10-20 Arabic words.
- long: 21-30 Arabic words.

Generate {n} texts for:
- topic: {topic}
- genre: {genre}
- length_bucket: {length_bucket}
- batch_id: {batch_id}
- created_at: {created_at}

Return valid CSV only. Do not add explanations, headings, markdown fences, numbering, or extra text.
"""

def call_gemini_batch(n, topic, genre, length_bucket, batch_id, created_at):
    prompt_text = GENERAL_PROMPT_TEMPLATE.format(
        n=n, topic=topic, genre=genre, length_bucket=length_bucket,
        batch_id=batch_id, created_at=created_at,
    )
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt_text)
    return prompt_text, response.text

def parse_csv_response(raw_text):
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        cleaned = "\n".join(lines)
    reader = csv.DictReader(io.StringIO(cleaned))
    raw_rows = list(reader)
    print("Rows parsed from response:", len(raw_rows))
    return raw_rows

def build_local_rows(raw_rows, topic, genre, length_bucket, batch_id, created_at):
    local_rows = []
    for i, row in enumerate(raw_rows, start=1):
        local_rows.append({
            "id": f"{batch_id}_{i:03d}", "text": row.get("text", "").strip(),
            "pool": "general", "topic": topic, "genre": genre, "length_bucket": length_bucket,
            "generator": "gemini", "generator_model_or_version": MODEL_NAME,
            "prompt_version": "v1.1", "batch_id": batch_id, "created_at": created_at,
            "audit_status": "pending", "cleaning_status": "pending", "notes": "",
        })
    return local_rows

BANNED_TERMS = [
    "بنك", "البنك", "مصرف", "حساب بنكي", "حساب مصرفي", "الحساب البنكي",
    "تحويل بنكي", "تحويل الأموال", "تحويل الفلوس", "رصيد الحساب", "رصيد البطاقة",
    "فيزا", "ماستر كارد", "بطاقة ائتمان", "بطاقة مدى", "بطاقة بنكية",
    "ابل باي", "Apple Pay", "STC Pay", "قرض", "فائدة القرض", "صراف آلي",
    "ATM", "الراجحي", "الأهلي", "سامبا", "الإنماء", "بنك ساب",
    "دفع الفاتورة", "فاتورة الحساب", "عمولة التحويل", "رسوم البنك",
]
PHONE_PATTERN = re.compile(r"(05\d{8}|\+9665\d{8})")
ID_PATTERN = re.compile(r"\b\d{10}\b")

def validate_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]
    for row in local_rows:
        text = row["text"]
        hard_reasons = []
        if not text: hard_reasons.append("blank_text")
        if text in seen_texts: hard_reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANNED_TERMS): hard_reasons.append("banking_term_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text): hard_reasons.append("possible_pii")
        if hard_reasons:
            row_copy = dict(row); row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy); continue
        row = dict(row)
        word_count = len(text.split())
        if not (min_words <= word_count <= max_words):
            row["notes"] = f"length_bucket_flag (actual words: {word_count})"
        seen_texts.add(text); passed.append(row)
    return passed, rejected

def build_general_batch_plan(total_target=12000, chunk_size=25):
    topics = [
        "Food and restaurants", "Travel and trips", "Study and university life",
        "Work and daily routines", "Sports and fitness", "Entertainment and hobbies",
        "Transport and traffic", "Technology and devices",
        "Shopping and products (non-financial)", "Family and social life",
        "Home and daily errands", "Weather and outdoor activities",
    ]
    genres = ["question", "statement", "comment"]
    length_pct = {"short": 0.35, "medium": 0.50, "long": 0.15}
    per_topic = total_target // len(topics)
    jobs = []
    job_counter = 1
    for topic in topics:
        base_genre = per_topic // len(genres)
        genre_remainder = per_topic - base_genre * len(genres)
        for g_idx, genre in enumerate(genres):
            genre_target = base_genre + (1 if g_idx < genre_remainder else 0)
            short_n = round(genre_target * length_pct["short"])
            medium_n = round(genre_target * length_pct["medium"])
            long_n = genre_target - short_n - medium_n
            for lb, count in {"short": short_n, "medium": medium_n, "long": long_n}.items():
                remaining = count
                while remaining > 0:
                    n = min(chunk_size, remaining)
                    jobs.append({"job_id": job_counter, "topic": topic, "genre": genre, "length_bucket": lb, "n": n})
                    job_counter += 1
                    remaining -= n
    return jobs

batch_plan = build_general_batch_plan(total_target=12000, chunk_size=25)

def run_general_production_batch(job_id, topic, genre, length_bucket, n, max_retries=2):
    batch_id = f"general_prod_{job_id:04d}_" + date.today().strftime("%Y%m%d")
    created_at = date.today().isoformat()
    attempt = 0
    while attempt <= max_retries:
        try:
            prompt_used, raw_response_text = call_gemini_batch(n, topic, genre, length_bucket, batch_id, created_at)
            raw_rows = parse_csv_response(raw_response_text)
            local_rows = build_local_rows(raw_rows, topic, genre, length_bucket, batch_id, created_at)
            passed_rows, rejected_rows = validate_rows(local_rows, length_bucket)
            break
        except Exception as e:
            attempt += 1
            print(f"Job {job_id} failed (attempt {attempt}): {e}")
            if attempt > max_retries:
                fail_folder = os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", f"general_batch_{job_id:04d}_FAILED")
                os.makedirs(fail_folder, exist_ok=True)
                with open(os.path.join(fail_folder, "error.json"), "w", encoding="utf-8") as f:
                    json.dump({"error": str(e), "topic": topic, "genre": genre, "length_bucket": length_bucket, "n": n}, f, ensure_ascii=False, indent=2)
                return {"batch_id": batch_id, "requested": n, "passed": 0, "rejected": 0, "failed": True}
            time.sleep(5)
    batch_folder = os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", f"general_batch_{job_id:04d}")
    os.makedirs(batch_folder, exist_ok=True)
    request_info = {"model": MODEL_NAME, "batch_id": batch_id, "topic": topic, "genre": genre, "length_bucket": length_bucket, "n_requested": n, "created_at": created_at, "prompt_used": prompt_used}
    with open(os.path.join(batch_folder, "request.json"), "w", encoding="utf-8") as f: json.dump(request_info, f, ensure_ascii=False, indent=2)
    with open(os.path.join(batch_folder, "raw_response.json"), "w", encoding="utf-8") as f: json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)
    local_fieldnames = list(local_rows[0].keys()) if local_rows else []
    if local_fieldnames:
        save_csv(local_rows, os.path.join(batch_folder, "parsed_rows.csv"), local_fieldnames)
        save_csv(passed_rows, os.path.join(batch_folder, "passed_rows.csv"), local_fieldnames)
        save_csv(rejected_rows, os.path.join(batch_folder, "rejected_rows.csv"), local_fieldnames + ["reject_reason"])
    validation_report = {"batch_id": batch_id, "requested": n, "parsed": len(local_rows), "passed": len(passed_rows), "rejected": len(rejected_rows), "rejection_reasons": [r.get("reject_reason", "") for r in rejected_rows], "failed": False}
    with open(os.path.join(batch_folder, "validation_report.json"), "w", encoding="utf-8") as f: json.dump(validation_report, f, ensure_ascii=False, indent=2)
    return validation_report

# 7. Banking prompt + functions
BANKING_PROMPT_TEMPLATE = """
You generate short, natural Saudi-style Arabic texts for a research corpus.

Goal: Create unlabeled Saudi-style banking-related text for a secondary continued pre-training experiment. These texts must not contain intent labels or copy examples from ArBanking77.

Output format: Return CSV rows only, with these columns:
id,text,pool,topic,genre,length_bucket,generator,prompt_version,batch_id,created_at,audit_status
Use `banking` for pool, `v1` for prompt_version, and `pending` for audit_status.

Content requirements:
- Write natural Saudi-style Arabic.
- Produce only one self-contained text per row.
- Use general banking situations such as cards, transfers, account access, payments, cash machines, or banking-app experience.
- Do not assign, mention, or imply an intent label.
- Do not copy or paraphrase any known ArBanking77 example.
- Do not include personal names, phone numbers, account numbers, addresses, IDs, or private information.
- Avoid repeated templates, repeated openings, and near-duplicate texts.
- Use generic banking concepts only. Do not mention real bank names, named card products, payment networks, loyalty programs, wallets, or product-specific fees, rewards, or benefits.

Genre guidance:
- question: a natural customer question.
- statement: a short customer observation.
- comment: a casual customer comment or app review.

Length guidance:
- short: 5-9 Arabic words.
- medium: 10-20 Arabic words.
- long: 21-30 Arabic words.

Generate {n} texts for:
- topic: {topic}
- genre: {genre}
- length_bucket: {length_bucket}
- batch_id: {batch_id}
- created_at: {created_at}

Return valid CSV only. Do not add explanations, headings, markdown fences, numbering, or extra text.
"""

def call_gemini_banking_batch(n, topic, genre, length_bucket, batch_id, created_at):
    prompt_text = BANKING_PROMPT_TEMPLATE.format(n=n, topic=topic, genre=genre, length_bucket=length_bucket, batch_id=batch_id, created_at=created_at)
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt_text)
    return prompt_text, response.text

def build_local_rows_banking(raw_rows, topic, genre, length_bucket, batch_id, created_at):
    local_rows = []
    for i, row in enumerate(raw_rows, start=1):
        local_rows.append({
            "id": f"{batch_id}_{i:03d}", "text": row.get("text", "").strip(),
            "pool": "banking", "topic": topic, "genre": genre, "length_bucket": length_bucket,
            "generator": "gemini", "generator_model_or_version": MODEL_NAME,
            "prompt_version": "v1", "batch_id": batch_id, "created_at": created_at,
            "audit_status": "pending", "cleaning_status": "pending", "notes": "",
        })
    return local_rows

BANKING_NAMED_TERMS = [
    "الراجحي", "الأهلي", "سامبا", "الإنماء", "ساب", "بنك ساب",
    "بنك الرياض", "بنك الجزيرة", "بنك البلاد", "الاستثمار", "الرياض المالية",
    "فيزا", "ماستر كارد", "مدى", "ابل باي", "Apple Pay", "STC Pay", "PayPal", "بايبال", "Visa", "Mastercard",
]

def validate_banking_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]
    for row in local_rows:
        text = row["text"]
        hard_reasons = []
        if not text: hard_reasons.append("blank_text")
        if text in seen_texts: hard_reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANKING_NAMED_TERMS): hard_reasons.append("named_bank_or_product_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text): hard_reasons.append("possible_pii")
        if hard_reasons:
            row_copy = dict(row); row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy); continue
        row = dict(row)
        word_count = len(text.split())
        if not (min_words <= word_count <= max_words):
            row["notes"] = f"length_bucket_flag (actual words: {word_count})"
        seen_texts.add(text); passed.append(row)
    return passed, rejected

def run_banking_production_batch(job_id, topic, genre, length_bucket, n, max_retries=2):
    batch_id = f"banking_prod_{job_id:04d}_" + date.today().strftime("%Y%m%d")
    created_at = date.today().isoformat()
    attempt = 0
    while attempt <= max_retries:
        try:
            prompt_used, raw_response_text = call_gemini_banking_batch(n, topic, genre, length_bucket, batch_id, created_at)
            raw_rows = parse_csv_response(raw_response_text)
            local_rows = build_local_rows_banking(raw_rows, topic, genre, length_bucket, batch_id, created_at)
            passed_rows, rejected_rows = validate_banking_rows(local_rows, length_bucket)
            break
        except Exception as e:
            attempt += 1
            print(f"Job {job_id} failed (attempt {attempt}): {e}")
            if attempt > max_retries:
                fail_folder = os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", f"banking_batch_{job_id:04d}_FAILED")
                os.makedirs(fail_folder, exist_ok=True)
                with open(os.path.join(fail_folder, "error.json"), "w", encoding="utf-8") as f:
                    json.dump({"error": str(e), "topic": topic, "genre": genre, "length_bucket": length_bucket, "n": n}, f, ensure_ascii=False, indent=2)
                return {"batch_id": batch_id, "requested": n, "passed": 0, "rejected": 0, "failed": True}
            time.sleep(5)
    batch_folder = os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", f"banking_batch_{job_id:04d}")
    os.makedirs(batch_folder, exist_ok=True)
    request_info = {"model": MODEL_NAME, "batch_id": batch_id, "topic": topic, "genre": genre, "length_bucket": length_bucket, "n_requested": n, "created_at": created_at, "prompt_used": prompt_used}
    with open(os.path.join(batch_folder, "request.json"), "w", encoding="utf-8") as f: json.dump(request_info, f, ensure_ascii=False, indent=2)
    with open(os.path.join(batch_folder, "raw_response.json"), "w", encoding="utf-8") as f: json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)
    local_fieldnames = list(local_rows[0].keys()) if local_rows else []
    if local_fieldnames:
        save_csv(local_rows, os.path.join(batch_folder, "parsed_rows.csv"), local_fieldnames)
        save_csv(passed_rows, os.path.join(batch_folder, "passed_rows.csv"), local_fieldnames)
        save_csv(rejected_rows, os.path.join(batch_folder, "rejected_rows.csv"), local_fieldnames + ["reject_reason"])
    validation_report = {"batch_id": batch_id, "requested": n, "parsed": len(local_rows), "passed": len(passed_rows), "rejected": len(rejected_rows), "rejection_reasons": [r.get("reject_reason", "") for r in rejected_rows], "failed": False}
    with open(os.path.join(batch_folder, "validation_report.json"), "w", encoding="utf-8") as f: json.dump(validation_report, f, ensure_ascii=False, indent=2)
    return validation_report

def build_banking_batch_plan(total_target=3000, chunk_size=25):
    topics = ["Money transfers and transfer status", "Debit and credit cards", "Account access and login", "Banking-app experience", "Payments and merchants", "Cash machines and cash withdrawal"]
    genres = ["question", "statement", "comment"]
    length_pct = {"short": 0.35, "medium": 0.50, "long": 0.15}
    per_topic = total_target // len(topics)
    jobs = []
    job_counter = 1
    for topic in topics:
        base_genre = per_topic // len(genres)
        genre_remainder = per_topic - base_genre * len(genres)
        for g_idx, genre in enumerate(genres):
            genre_target = base_genre + (1 if g_idx < genre_remainder else 0)
            short_n = round(genre_target * length_pct["short"])
            medium_n = round(genre_target * length_pct["medium"])
            long_n = genre_target - short_n - medium_n
            for lb, count in {"short": short_n, "medium": medium_n, "long": long_n}.items():
                remaining = count
                while remaining > 0:
                    n = min(chunk_size, remaining)
                    jobs.append({"job_id": job_counter, "topic": topic, "genre": genre, "length_bucket": lb, "n": n})
                    job_counter += 1
                    remaining -= n
    return jobs

banking_batch_plan = build_banking_batch_plan(total_target=3000, chunk_size=25)

print("=== SETUP COMPLETE ===")
print("batch_plan jobs:", len(batch_plan))
print("banking_batch_plan jobs:", len(banking_batch_plan))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== SETUP COMPLETE ===
batch_plan jobs: 504
banking_batch_plan jobs: 144


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT"

print("Path exists:", os.path.exists(PROJECT_ROOT))
print("Folder contents:")
print(os.listdir(PROJECT_ROOT))

Path exists: True
Folder contents:
['00_admin', '01_raw_data', '02_processed_data', '03_synthetic_data', '04_audits', '05_runs', '06_models_and_checkpoints', '07_results', '08_figures', '09_presentations', '10_archive']


In [ ]:
SYNTHETIC_ROOT = os.path.join(PROJECT_ROOT, "03_synthetic_data")

print("Path exists:", os.path.exists(SYNTHETIC_ROOT))
print("Folder contents:")
print(os.listdir(SYNTHETIC_ROOT))

Path exists: True
Folder contents:
[]


In [ ]:
subfolders = [
    "00_pilot_raw",
    "01_api_smoke_batches",
    "02_general_raw_batches",
    "03_banking_raw_batches",
    "04_cleaning_logs",
    "05_final_corpora",
    "06_final_audit",
]

for folder in subfolders:
    folder_path = os.path.join(SYNTHETIC_ROOT, folder)
    os.makedirs(folder_path, exist_ok=True)

print("Created folders:")
print(sorted(os.listdir(SYNTHETIC_ROOT)))

Created folders:
['00_pilot_raw', '01_api_smoke_batches', '02_general_raw_batches', '03_banking_raw_batches', '04_cleaning_logs', '05_final_corpora', '06_final_audit']


In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 19.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.5-flash-lite"

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Reply with exactly one word: OK"
)

print("Connection successful. Model replied:", response.text)

Connection successful. Model replied: OK


In [ ]:
GENERAL_PROMPT_TEMPLATE = """
You generate short, natural Saudi-style Arabic texts for a research corpus.

Goal: Create unlabeled general Saudi Arabic text for continued pre-training. The text will be used only to expose a language model to Saudi-style wording before a separate banking intent-classification task.

Output format: Return CSV rows only, with these columns:
id,text,pool,topic,genre,length_bucket,generator,prompt_version,batch_id,created_at,audit_status
Use `general` for pool, `v1.1` for prompt_version, and `pending` for audit_status.

Content requirements:
- Write natural Saudi-style Arabic, not formal MSA and not deliberately broken Arabic.
- Produce only one self-contained text per row.
- Use the requested topic, genre, and length bucket.
- Use everyday, non-financial topics only.
- Do not include personal names, phone numbers, account numbers, addresses, IDs, or private information.
- Avoid repeated templates, repeated openings, and near-duplicate texts.
- Use globally unique IDs with the batch prefix, for example: general_pilot_01_001.
- Avoid ambiguous standalone words that may be banking-related, such as "branch", unless the non-financial context is explicit in the same text.

Strict exclusions: Do NOT mention or imply banks, banking applications, cards, transfers, balances, payments, invoices, money, currencies, loans, accounts, ATMs, or any banking intent from ArBanking77.

Genre guidance:
- question: a natural question someone may ask in daily life.
- statement: a short everyday statement or observation.
- comment: a casual comment or review about a non-financial everyday topic.

Length guidance:
- short: 5-9 Arabic words.
- medium: 10-20 Arabic words.
- long: 21-30 Arabic words.

Generate {n} texts for:
- topic: {topic}
- genre: {genre}
- length_bucket: {length_bucket}
- batch_id: {batch_id}
- created_at: {created_at}

Return valid CSV only. Do not add explanations, headings, markdown fences, numbering, or extra text.
"""

In [ ]:
def call_gemini_batch(n, topic, genre, length_bucket, batch_id, created_at):
    prompt_text = GENERAL_PROMPT_TEMPLATE.format(
        n=n, topic=topic, genre=genre,
        length_bucket=length_bucket,
        batch_id=batch_id, created_at=created_at,
    )
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
    )
    return prompt_text, response.text

In [ ]:
from datetime import date

BATCH_ID = "general_smoke_001_" + date.today().strftime("%Y%m%d")
TOPIC = "Food and restaurants"
GENRE = "question"
LENGTH_BUCKET = "short"
N_TEXTS = 25
CREATED_AT = date.today().isoformat()

prompt_used, raw_response_text = call_gemini_batch(
    n=N_TEXTS,
    topic=TOPIC,
    genre=GENRE,
    length_bucket=LENGTH_BUCKET,
    batch_id=BATCH_ID,
    created_at=CREATED_AT,
)

print("Batch ID:", BATCH_ID)
print("Raw response preview:")
print(raw_response_text[:500])

Batch ID: general_smoke_001_20260829
Raw response preview:
id,text,pool,topic,genre,length_bucket,generator,prompt_version,batch_id,created_at,audit_status
general_smoke_001_20260829_001,وش أفضل مطعم كشري بالرياض حالياً؟,general,Food and restaurants,question,short,llm,v1.1,general_smoke_001_20260829,2026-08-29,pending
general_smoke_001_20260829_002,وين ألقى ألذ كيكة شوكولاتة بالحي؟,general,Food and restaurants,question,short,llm,v1.1,general_smoke_001_20260829,2026-08-29,pending
general_smoke_001_20260829_003,كم سعر حبة البرجر العادي عندهم؟,general,Food


In [ ]:
import csv
import io

def parse_csv_response(raw_text):
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        cleaned = "\n".join(lines)

    reader = csv.DictReader(io.StringIO(cleaned))
    raw_rows = list(reader)
    print("Rows parsed from response:", len(raw_rows))
    return raw_rows

raw_rows = parse_csv_response(raw_response_text)
raw_rows[:3]

Rows parsed from response: 25


[{'id': 'general_smoke_001_20260829_001',
  'text': 'وش أفضل مطعم كشري بالرياض حالياً؟',
  'pool': 'general',
  'topic': 'Food and restaurants',
  'genre': 'question',
  'length_bucket': 'short',
  'generator': 'llm',
  'prompt_version': 'v1.1',
  'batch_id': 'general_smoke_001_20260829',
  'created_at': '2026-08-29',
  'audit_status': 'pending'},
 {'id': 'general_smoke_001_20260829_002',
  'text': 'وين ألقى ألذ كيكة شوكولاتة بالحي؟',
  'pool': 'general',
  'topic': 'Food and restaurants',
  'genre': 'question',
  'length_bucket': 'short',
  'generator': 'llm',
  'prompt_version': 'v1.1',
  'batch_id': 'general_smoke_001_20260829',
  'created_at': '2026-08-29',
  'audit_status': 'pending'},
 {'id': 'general_smoke_001_20260829_003',
  'text': 'كم سعر حبة البرجر العادي عندهم؟',
  'pool': 'general',
  'topic': 'Food and restaurants',
  'genre': 'question',
  'length_bucket': 'short',
  'generator': 'llm',
  'prompt_version': 'v1.1',
  'batch_id': 'general_smoke_001_20260829',
  'created_a

In [ ]:
def build_local_rows(raw_rows, topic, genre, length_bucket, batch_id, created_at):
    local_rows = []
    for i, row in enumerate(raw_rows, start=1):
        local_rows.append({
            "id": f"{batch_id}_{i:03d}",
            "text": row.get("text", "").strip(),
            "pool": "general",
            "topic": topic,
            "genre": genre,
            "length_bucket": length_bucket,
            "generator": "gemini",
            "generator_model_or_version": MODEL_NAME,
            "prompt_version": "v1.1",
            "batch_id": batch_id,
            "created_at": created_at,
            "audit_status": "pending",
            "cleaning_status": "pending",
            "notes": "",
        })
    return local_rows

local_rows = build_local_rows(
    raw_rows, topic=TOPIC, genre=GENRE, length_bucket=LENGTH_BUCKET,
    batch_id=BATCH_ID, created_at=CREATED_AT,
)
local_rows[:2]

[{'id': 'general_smoke_001_20260829_001',
  'text': 'وش أفضل مطعم كشري بالرياض حالياً؟',
  'pool': 'general',
  'topic': 'Food and restaurants',
  'genre': 'question',
  'length_bucket': 'short',
  'generator': 'gemini',
  'generator_model_or_version': 'gemini-3.5-flash-lite',
  'prompt_version': 'v1.1',
  'batch_id': 'general_smoke_001_20260829',
  'created_at': '2026-08-29',
  'audit_status': 'pending',
  'cleaning_status': 'pending',
  'notes': ''},
 {'id': 'general_smoke_001_20260829_002',
  'text': 'وين ألقى ألذ كيكة شوكولاتة بالحي؟',
  'pool': 'general',
  'topic': 'Food and restaurants',
  'genre': 'question',
  'length_bucket': 'short',
  'generator': 'gemini',
  'generator_model_or_version': 'gemini-3.5-flash-lite',
  'prompt_version': 'v1.1',
  'batch_id': 'general_smoke_001_20260829',
  'created_at': '2026-08-29',
  'audit_status': 'pending',
  'cleaning_status': 'pending',
  'notes': ''}]

In [ ]:
import re

BANNED_TERMS = [
    "بنك", "البنك", "مصرف", "حساب", "تحويل", "رصيد", "فيزا", "ماستر كارد",
    "مدى", "ابل باي", "Apple Pay", "STC Pay", "قرض", "فائدة", "صراف",
    "ATM", "بطاقة ائتمان", "بطاقة مدى", "الراجحي", "الأهلي", "سامبا",
    "الإنماء", "ساب", "بطاقة بنكية", "دفع", "فاتورة", "عمولة",
]

PHONE_PATTERN = re.compile(r"(05\d{8}|\+9665\d{8})")
ID_PATTERN = re.compile(r"\b\d{10}\b")

def validate_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]

    for row in local_rows:
        text = row["text"]
        reasons = []
        if not text:
            reasons.append("blank_text")
        if text in seen_texts:
            reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANNED_TERMS):
            reasons.append("banking_term_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text):
            reasons.append("possible_pii")
        word_count = len(text.split())
        if not (min_words - 2 <= word_count <= max_words + 3):
            reasons.append("length_bucket_flag")

        if reasons:
            row_copy = dict(row)
            row_copy["reject_reason"] = ";".join(reasons)
            rejected.append(row_copy)
        else:
            seen_texts.add(text)
            passed.append(row)

    return passed, rejected

passed_rows, rejected_rows = validate_rows(local_rows, LENGTH_BUCKET)
print("Passed:", len(passed_rows))
print("Rejected:", len(rejected_rows))
for r in rejected_rows:
    print(r["id"], "-", r["reject_reason"])

Passed: 25
Rejected: 0


In [ ]:
import json
import csv
import os

BATCH_FOLDER = os.path.join(SYNTHETIC_ROOT, "01_api_smoke_batches", "general_smoke_001")
os.makedirs(BATCH_FOLDER, exist_ok=True)

def save_csv(rows, filepath, fieldnames):
    with open(filepath, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

# 1. the exact request sent
request_info = {
    "model": MODEL_NAME,
    "batch_id": BATCH_ID,
    "topic": TOPIC,
    "genre": GENRE,
    "length_bucket": LENGTH_BUCKET,
    "n_requested": N_TEXTS,
    "created_at": CREATED_AT,
    "prompt_used": prompt_used,
}
with open(os.path.join(BATCH_FOLDER, "request.json"), "w", encoding="utf-8") as f:
    json.dump(request_info, f, ensure_ascii=False, indent=2)

# 2. the raw text Gemini returned, untouched
with open(os.path.join(BATCH_FOLDER, "raw_response.json"), "w", encoding="utf-8") as f:
    json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)

# 3. all rows after adding local metadata
local_fieldnames = list(local_rows[0].keys())
save_csv(local_rows, os.path.join(BATCH_FOLDER, "parsed_rows.csv"), local_fieldnames)

# 4. only the clean rows (what could go into the final corpus later)
save_csv(passed_rows, os.path.join(BATCH_FOLDER, "passed_rows.csv"), local_fieldnames)

# 5. rejected rows with reasons (empty file is fine if nothing rejected)
rejected_fieldnames = local_fieldnames + ["reject_reason"]
save_csv(rejected_rows, os.path.join(BATCH_FOLDER, "rejected_rows.csv"), rejected_fieldnames)

# 6. summary counts
validation_report = {
    "batch_id": BATCH_ID,
    "requested": N_TEXTS,
    "parsed": len(local_rows),
    "passed": len(passed_rows),
    "rejected": len(rejected_rows),
    "rejection_reasons": [r["reject_reason"] for r in rejected_rows],
}
with open(os.path.join(BATCH_FOLDER, "validation_report.json"), "w", encoding="utf-8") as f:
    json.dump(validation_report, f, ensure_ascii=False, indent=2)

print("Saved batch folder:", BATCH_FOLDER)
print("Files saved:", os.listdir(BATCH_FOLDER))

Saved batch folder: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/01_api_smoke_batches/general_smoke_001
Files saved: ['request.json', 'raw_response.json', 'parsed_rows.csv', 'passed_rows.csv', 'rejected_rows.csv', 'validation_report.json']


In [ ]:
def run_general_smoke_batch(batch_number, topic, genre, length_bucket, n=25):
    batch_id = f"general_smoke_{batch_number:03d}_" + date.today().strftime("%Y%m%d")
    created_at = date.today().isoformat()

    prompt_used, raw_response_text = call_gemini_batch(
        n=n, topic=topic, genre=genre, length_bucket=length_bucket,
        batch_id=batch_id, created_at=created_at,
    )
    raw_rows = parse_csv_response(raw_response_text)
    local_rows = build_local_rows(raw_rows, topic, genre, length_bucket, batch_id, created_at)
    passed_rows, rejected_rows = validate_rows(local_rows, length_bucket)

    batch_folder = os.path.join(SYNTHETIC_ROOT, "01_api_smoke_batches", f"general_smoke_{batch_number:03d}")
    os.makedirs(batch_folder, exist_ok=True)

    request_info = {
        "model": MODEL_NAME, "batch_id": batch_id, "topic": topic, "genre": genre,
        "length_bucket": length_bucket, "n_requested": n, "created_at": created_at,
        "prompt_used": prompt_used,
    }
    with open(os.path.join(batch_folder, "request.json"), "w", encoding="utf-8") as f:
        json.dump(request_info, f, ensure_ascii=False, indent=2)
    with open(os.path.join(batch_folder, "raw_response.json"), "w", encoding="utf-8") as f:
        json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)

    local_fieldnames = list(local_rows[0].keys())
    save_csv(local_rows, os.path.join(batch_folder, "parsed_rows.csv"), local_fieldnames)
    save_csv(passed_rows, os.path.join(batch_folder, "passed_rows.csv"), local_fieldnames)
    save_csv(rejected_rows, os.path.join(batch_folder, "rejected_rows.csv"), local_fieldnames + ["reject_reason"])

    validation_report = {
        "batch_id": batch_id, "requested": n, "parsed": len(local_rows),
        "passed": len(passed_rows), "rejected": len(rejected_rows),
        "rejection_reasons": [r["reject_reason"] for r in rejected_rows],
    }
    with open(os.path.join(batch_folder, "validation_report.json"), "w", encoding="utf-8") as f:
        json.dump(validation_report, f, ensure_ascii=False, indent=2)

    print(f"Batch {batch_id}: requested={n}, passed={len(passed_rows)}, rejected={len(rejected_rows)}")
    return validation_report

# Smoke batch 002: different topic/genre/length to broaden the test
report_002 = run_general_smoke_batch(
    batch_number=2, topic="Travel and trips", genre="statement", length_bucket="medium", n=25
)

Rows parsed from response: 25
Batch general_smoke_002_20260829: requested=25, passed=25, rejected=0


In [ ]:
report_003 = run_general_smoke_batch(
    batch_number=3, topic="Technology and devices", genre="comment", length_bucket="long", n=25
)

Rows parsed from response: 25
Batch general_smoke_003_20260829: requested=25, passed=25, rejected=0


In [ ]:
# Diagnostic run only — does not save anything
debug_topic = "Technology and devices"
debug_genre = "comment"
debug_length = "long"

_, debug_raw_text = call_gemini_batch(
    n=25, topic=debug_topic, genre=debug_genre, length_bucket=debug_length,
    batch_id="debug_run", created_at=date.today().isoformat(),
)
debug_raw_rows = parse_csv_response(debug_raw_text)
debug_local_rows = build_local_rows(
    debug_raw_rows, debug_topic, debug_genre, debug_length,
    "debug_run", date.today().isoformat(),
)
debug_passed, debug_rejected = validate_rows(debug_local_rows, debug_length)

print("Passed:", len(debug_passed), "Rejected:", len(debug_rejected))
for r in debug_rejected:
    print(r["reject_reason"], "→", r["text"])

Rows parsed from response: 25
Passed: 22 Rejected: 3
banking_term_leakage → الساعات الذكية صارت تفيد الواحد مرة خصوصاً بمتابعة الرياضة وحساب الخطوات، بس حسافة شحنها يخلص بسرعة إذا شغلت كل الخدمات.
length_bucket_flag → الماوس والكيبورد اللاسلكية شكل المكتب صار مرتب بدون خيوط وزحمة، وتحس الشغل صار أريح بكثير مقارنة بالأجهزة القديمة.
length_bucket_flag → الروبوتات الصغيرة اللي تنظف البيت بروحه راحة نفسية، تخليه يشتغل وأنت قاعد تتقهوى، بس عيبه صوته مزعج شوي.


In [ ]:
BANNED_TERMS = [
    "بنك", "البنك", "مصرف", "حساب بنكي", "حساب مصرفي", "الحساب البنكي",
    "تحويل بنكي", "تحويل الأموال", "تحويل الفلوس", "رصيد الحساب", "رصيد البطاقة",
    "فيزا", "ماستر كارد", "بطاقة ائتمان", "بطاقة مدى", "بطاقة بنكية",
    "ابل باي", "Apple Pay", "STC Pay", "قرض", "فائدة القرض", "صراف آلي",
    "ATM", "الراجحي", "الأهلي", "سامبا", "الإنماء", "بنك ساب",
    "دفع الفاتورة", "فاتورة الحساب", "عمولة التحويل", "رسوم البنك",
]

def validate_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]

    for row in local_rows:
        text = row["text"]
        hard_reasons = []

        if not text:
            hard_reasons.append("blank_text")
        if text in seen_texts:
            hard_reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANNED_TERMS):
            hard_reasons.append("banking_term_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text):
            hard_reasons.append("possible_pii")

        if hard_reasons:
            row_copy = dict(row)
            row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy)
            continue

        row = dict(row)
        word_count = len(text.split())
        if not (min_words <= word_count <= max_words):
            row["notes"] = f"length_bucket_flag (actual words: {word_count})"

        seen_texts.add(text)
        passed.append(row)

    return passed, rejected

In [ ]:
def build_general_batch_plan(total_target=12000, chunk_size=25):
    topics = [
        "Food and restaurants", "Travel and trips", "Study and university life",
        "Work and daily routines", "Sports and fitness", "Entertainment and hobbies",
        "Transport and traffic", "Technology and devices",
        "Shopping and products (non-financial)", "Family and social life",
        "Home and daily errands", "Weather and outdoor activities",
    ]
    genres = ["question", "statement", "comment"]
    length_pct = {"short": 0.35, "medium": 0.50, "long": 0.15}

    per_topic = total_target // len(topics)
    jobs = []
    job_counter = 1

    for topic in topics:
        base_genre = per_topic // len(genres)
        genre_remainder = per_topic - base_genre * len(genres)
        for g_idx, genre in enumerate(genres):
            genre_target = base_genre + (1 if g_idx < genre_remainder else 0)

            short_n = round(genre_target * length_pct["short"])
            medium_n = round(genre_target * length_pct["medium"])
            long_n = genre_target - short_n - medium_n
            length_counts = {"short": short_n, "medium": medium_n, "long": long_n}

            for lb, count in length_counts.items():
                remaining = count
                while remaining > 0:
                    n = min(chunk_size, remaining)
                    jobs.append({"job_id": job_counter, "topic": topic, "genre": genre, "length_bucket": lb, "n": n})
                    job_counter += 1
                    remaining -= n
    return jobs

batch_plan = build_general_batch_plan(total_target=12000, chunk_size=25)
print("Total batches planned:", len(batch_plan))
print("Total texts targeted:", sum(j["n"] for j in batch_plan))

Total batches planned: 504
Total texts targeted: 12000


In [ ]:
import time

def run_general_production_batch(job_id, topic, genre, length_bucket, n, max_retries=2):
    batch_id = f"general_prod_{job_id:04d}_" + date.today().strftime("%Y%m%d")
    created_at = date.today().isoformat()

    attempt = 0
    while attempt <= max_retries:
        try:
            prompt_used, raw_response_text = call_gemini_batch(
                n=n, topic=topic, genre=genre, length_bucket=length_bucket,
                batch_id=batch_id, created_at=created_at,
            )
            raw_rows = parse_csv_response(raw_response_text)
            local_rows = build_local_rows(raw_rows, topic, genre, length_bucket, batch_id, created_at)
            passed_rows, rejected_rows = validate_rows(local_rows, length_bucket)
            break
        except Exception as e:
            attempt += 1
            print(f"Job {job_id} failed (attempt {attempt}): {e}")
            if attempt > max_retries:
                fail_folder = os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", f"general_batch_{job_id:04d}_FAILED")
                os.makedirs(fail_folder, exist_ok=True)
                with open(os.path.join(fail_folder, "error.json"), "w", encoding="utf-8") as f:
                    json.dump({"error": str(e), "topic": topic, "genre": genre, "length_bucket": length_bucket, "n": n}, f, ensure_ascii=False, indent=2)
                return {"batch_id": batch_id, "requested": n, "passed": 0, "rejected": 0, "failed": True}
            time.sleep(5)

    batch_folder = os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", f"general_batch_{job_id:04d}")
    os.makedirs(batch_folder, exist_ok=True)

    request_info = {
        "model": MODEL_NAME, "batch_id": batch_id, "topic": topic, "genre": genre,
        "length_bucket": length_bucket, "n_requested": n, "created_at": created_at,
        "prompt_used": prompt_used,
    }
    with open(os.path.join(batch_folder, "request.json"), "w", encoding="utf-8") as f:
        json.dump(request_info, f, ensure_ascii=False, indent=2)
    with open(os.path.join(batch_folder, "raw_response.json"), "w", encoding="utf-8") as f:
        json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)

    local_fieldnames = list(local_rows[0].keys()) if local_rows else []
    if local_fieldnames:
        save_csv(local_rows, os.path.join(batch_folder, "parsed_rows.csv"), local_fieldnames)
        save_csv(passed_rows, os.path.join(batch_folder, "passed_rows.csv"), local_fieldnames)
        save_csv(rejected_rows, os.path.join(batch_folder, "rejected_rows.csv"), local_fieldnames + ["reject_reason"])

    validation_report = {
        "batch_id": batch_id, "requested": n, "parsed": len(local_rows),
        "passed": len(passed_rows), "rejected": len(rejected_rows),
        "rejection_reasons": [r.get("reject_reason", "") for r in rejected_rows],
        "failed": False,
    }
    with open(os.path.join(batch_folder, "validation_report.json"), "w", encoding="utf-8") as f:
        json.dump(validation_report, f, ensure_ascii=False, indent=2)

    return validation_report

In [ ]:
total_passed = 0
total_requested = 0
failed_jobs = []

for job in batch_plan:
    result = run_general_production_batch(
        job_id=job["job_id"],
        topic=job["topic"],
        genre=job["genre"],
        length_bucket=job["length_bucket"],
        n=job["n"],
    )
    total_requested += job["n"]
    total_passed += result["passed"]
    if result.get("failed"):
        failed_jobs.append(job)

    if job["job_id"] % 20 == 0:
        print(f"Progress: job {job['job_id']}/{len(batch_plan)} | requested so far={total_requested} | passed so far={total_passed} | failed batches={len(failed_jobs)}")

    time.sleep(1.5)

print("=== DONE ===")
print("Total requested:", total_requested)
print("Total passed:", total_passed)
print("Failed batches:", len(failed_jobs))

Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 17
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 17
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 17
Rows parsed from response: 25
Progress: job 20/504 | requested so far=476 | passed so far=476 | failed batches=0
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 16
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from response: 25
Rows parsed from 

In [ ]:
import glob

report_files = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "*", "validation_report.json"))

total_requested_final = 0
total_passed_final = 0
total_failed_batches = 0

for rf in report_files:
    with open(rf, "r", encoding="utf-8") as f:
        report = json.load(f)
    total_requested_final += report.get("requested", 0)
    total_passed_final += report.get("passed", 0)
    if report.get("failed"):
        total_failed_batches += 1

print("Batches saved:", len(report_files))
print("Total requested:", total_requested_final)
print("Total passed (clean):", total_passed_final)
print("Failed batches:", total_failed_batches)
print("Percent of 12,000 target:", round(100 * total_passed_final / 12000, 1), "%")

Batches saved: 493
Total requested: 11742
Total passed (clean): 11538
Failed batches: 0
Percent of 12,000 target: 96.2 %


---
## Section 5 — General Production: Status

**Result:** 11,538 / 12,000 clean texts generated and saved (96.2%).
**Blocked by:** Free-tier daily API quota exhausted (11 batches pending, ~460 texts).
**Plan:** Remaining general batches + all banking production resume after quota
reset (~10am tomorrow, well within the fresh daily limit).

---
## Section 6 — Nested Subsets (general_3k / general_6k)

Draft subsets built from currently available clean data (pre-freeze — final
freeze happens after Sunday's full cleaning pipeline: cross-batch dedup,
Saudi-test leakage check, and manual audit).
---

In [ ]:
import glob
import hashlib

batch_folders = sorted(glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "*")))
all_general_rows = []

for folder in batch_folders:
    passed_csv = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_csv):
        with open(passed_csv, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            all_general_rows.extend(list(reader))

print("Total clean rows collected from all batches:", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
cross_batch_duplicates = []

for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        cross_batch_duplicates.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("Cross-batch exact duplicates removed:", len(cross_batch_duplicates))
print("Remaining clean unique rows:", len(deduped_rows))

CLEANING_LOG_FOLDER = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs")
os.makedirs(CLEANING_LOG_FOLDER, exist_ok=True)
dedup_log_path = os.path.join(CLEANING_LOG_FOLDER, "general_cross_batch_dedup_log.csv")
log_fieldnames = list(all_general_rows[0].keys())
save_csv(cross_batch_duplicates, dedup_log_path, log_fieldnames)
print("Deletion log saved:", dedup_log_path)

Total clean rows collected from all batches: 11538
Cross-batch exact duplicates removed: 40
Remaining clean unique rows: 11498
Deletion log saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/general_cross_batch_dedup_log.csv


In [ ]:
FIXED_SEED = "sarf_general_corpus_v1"

def stable_sort_key(row_id):
    return hashlib.sha256(f"{FIXED_SEED}_{row_id}".encode("utf-8")).hexdigest()

deduped_rows.sort(key=lambda r: stable_sort_key(r["id"]))

general_3k = deduped_rows[:3000]
general_6k = deduped_rows[:6000]
general_12k_so_far = deduped_rows[:12000]

print("3k available:", len(general_3k))
print("6k available:", len(general_6k))
print("12k available now:", len(general_12k_so_far), "(final target: 12000)")

FINAL_FOLDER = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
os.makedirs(FINAL_FOLDER, exist_ok=True)

fieldnames = list(deduped_rows[0].keys())
save_csv(general_3k, os.path.join(FINAL_FOLDER, "general_3k_draft.csv"), fieldnames)
save_csv(general_6k, os.path.join(FINAL_FOLDER, "general_6k_draft.csv"), fieldnames)

print("Saved to:", FINAL_FOLDER)
print(os.listdir(FINAL_FOLDER))

3k available: 3000
6k available: 6000
12k available now: 11498 (final target: 12000)
Saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/05_final_corpora
['general_3k_draft.csv', 'general_6k_draft.csv']


---
## Section 7 — Banking Prep (ready to run tomorrow after quota reset)
## Section 8 — Finish remaining General batches (retry only the 11 failed jobs)
---

In [ ]:
BANKING_PROMPT_TEMPLATE = """
You generate short, natural Saudi-style Arabic texts for a research corpus.

Goal: Create unlabeled Saudi-style banking-related text for a secondary continued pre-training experiment. These texts must not contain intent labels or copy examples from ArBanking77.

Output format: Return CSV rows only, with these columns:
id,text,pool,topic,genre,length_bucket,generator,prompt_version,batch_id,created_at,audit_status
Use `banking` for pool, `v1` for prompt_version, and `pending` for audit_status.

Content requirements:
- Write natural Saudi-style Arabic.
- Produce only one self-contained text per row.
- Use general banking situations such as cards, transfers, account access, payments, cash machines, or banking-app experience.
- Do not assign, mention, or imply an intent label.
- Do not copy or paraphrase any known ArBanking77 example.
- Do not include personal names, phone numbers, account numbers, addresses, IDs, or private information.
- Avoid repeated templates, repeated openings, and near-duplicate texts.
- Use generic banking concepts only. Do not mention real bank names, named card products, payment networks, loyalty programs, wallets, or product-specific fees, rewards, or benefits.

Genre guidance:
- question: a natural customer question.
- statement: a short customer observation.
- comment: a casual customer comment or app review.

Length guidance:
- short: 5-9 Arabic words.
- medium: 10-20 Arabic words.
- long: 21-30 Arabic words.

Generate {n} texts for:
- topic: {topic}
- genre: {genre}
- length_bucket: {length_bucket}
- batch_id: {batch_id}
- created_at: {created_at}

Return valid CSV only. Do not add explanations, headings, markdown fences, numbering, or extra text.
"""

def call_gemini_banking_batch(n, topic, genre, length_bucket, batch_id, created_at):
    prompt_text = BANKING_PROMPT_TEMPLATE.format(
        n=n, topic=topic, genre=genre, length_bucket=length_bucket,
        batch_id=batch_id, created_at=created_at,
    )
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt_text)
    return prompt_text, response.text

def build_local_rows_banking(raw_rows, topic, genre, length_bucket, batch_id, created_at):
    local_rows = []
    for i, row in enumerate(raw_rows, start=1):
        local_rows.append({
            "id": f"{batch_id}_{i:03d}",
            "text": row.get("text", "").strip(),
            "pool": "banking",
            "topic": topic,
            "genre": genre,
            "length_bucket": length_bucket,
            "generator": "gemini",
            "generator_model_or_version": MODEL_NAME,
            "prompt_version": "v1",
            "batch_id": batch_id,
            "created_at": created_at,
            "audit_status": "pending",
            "cleaning_status": "pending",
            "notes": "",
        })
    return local_rows

# Named/branded terms forbidden in BOTH pools — but here generic banking
# words (بنك، حساب، تحويل، بطاقة، رصيد...) are ALLOWED, since this is the
# banking pool itself. Only specific real names/brands are rejected.
BANKING_NAMED_TERMS = [
    "الراجحي", "الأهلي", "سامبا", "الإنماء", "ساب", "بنك ساب",
    "بنك الرياض", "بنك الجزيرة", "بنك البلاد", "الاستثمار", "الرياض المالية",
    "فيزا", "ماستر كارد", "مدى",
    "ابل باي", "Apple Pay", "STC Pay", "PayPal", "بايبال", "Visa", "Mastercard",
]

def validate_banking_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]

    for row in local_rows:
        text = row["text"]
        hard_reasons = []
        if not text:
            hard_reasons.append("blank_text")
        if text in seen_texts:
            hard_reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANKING_NAMED_TERMS):
            hard_reasons.append("named_bank_or_product_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text):
            hard_reasons.append("possible_pii")

        if hard_reasons:
            row_copy = dict(row)
            row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy)
            continue

        row = dict(row)
        word_count = len(text.split())
        if not (min_words <= word_count <= max_words):
            row["notes"] = f"length_bucket_flag (actual words: {word_count})"
        seen_texts.add(text)
        passed.append(row)

    return passed, rejected

def run_banking_production_batch(job_id, topic, genre, length_bucket, n, max_retries=2):
    batch_id = f"banking_prod_{job_id:04d}_" + date.today().strftime("%Y%m%d")
    created_at = date.today().isoformat()
    attempt = 0
    while attempt <= max_retries:
        try:
            prompt_used, raw_response_text = call_gemini_banking_batch(
                n=n, topic=topic, genre=genre, length_bucket=length_bucket,
                batch_id=batch_id, created_at=created_at,
            )
            raw_rows = parse_csv_response(raw_response_text)
            local_rows = build_local_rows_banking(raw_rows, topic, genre, length_bucket, batch_id, created_at)
            passed_rows, rejected_rows = validate_banking_rows(local_rows, length_bucket)
            break
        except Exception as e:
            attempt += 1
            print(f"Job {job_id} failed (attempt {attempt}): {e}")
            if attempt > max_retries:
                fail_folder = os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", f"banking_batch_{job_id:04d}_FAILED")
                os.makedirs(fail_folder, exist_ok=True)
                with open(os.path.join(fail_folder, "error.json"), "w", encoding="utf-8") as f:
                    json.dump({"error": str(e), "topic": topic, "genre": genre, "length_bucket": length_bucket, "n": n}, f, ensure_ascii=False, indent=2)
                return {"batch_id": batch_id, "requested": n, "passed": 0, "rejected": 0, "failed": True}
            time.sleep(5)

    batch_folder = os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", f"banking_batch_{job_id:04d}")
    os.makedirs(batch_folder, exist_ok=True)
    request_info = {
        "model": MODEL_NAME, "batch_id": batch_id, "topic": topic, "genre": genre,
        "length_bucket": length_bucket, "n_requested": n, "created_at": created_at,
        "prompt_used": prompt_used,
    }
    with open(os.path.join(batch_folder, "request.json"), "w", encoding="utf-8") as f:
        json.dump(request_info, f, ensure_ascii=False, indent=2)
    with open(os.path.join(batch_folder, "raw_response.json"), "w", encoding="utf-8") as f:
        json.dump({"raw_text": raw_response_text}, f, ensure_ascii=False, indent=2)

    local_fieldnames = list(local_rows[0].keys()) if local_rows else []
    if local_fieldnames:
        save_csv(local_rows, os.path.join(batch_folder, "parsed_rows.csv"), local_fieldnames)
        save_csv(passed_rows, os.path.join(batch_folder, "passed_rows.csv"), local_fieldnames)
        save_csv(rejected_rows, os.path.join(batch_folder, "rejected_rows.csv"), local_fieldnames + ["reject_reason"])

    validation_report = {
        "batch_id": batch_id, "requested": n, "parsed": len(local_rows),
        "passed": len(passed_rows), "rejected": len(rejected_rows),
        "rejection_reasons": [r.get("reject_reason", "") for r in rejected_rows],
        "failed": False,
    }
    with open(os.path.join(batch_folder, "validation_report.json"), "w", encoding="utf-8") as f:
        json.dump(validation_report, f, ensure_ascii=False, indent=2)
    return validation_report

In [ ]:
def build_banking_batch_plan(total_target=3000, chunk_size=25):
    topics = [
        "Money transfers and transfer status", "Debit and credit cards",
        "Account access and login", "Banking-app experience",
        "Payments and merchants", "Cash machines and cash withdrawal",
    ]
    genres = ["question", "statement", "comment"]
    length_pct = {"short": 0.35, "medium": 0.50, "long": 0.15}
    per_topic = total_target // len(topics)
    jobs = []
    job_counter = 1
    for topic in topics:
        base_genre = per_topic // len(genres)
        genre_remainder = per_topic - base_genre * len(genres)
        for g_idx, genre in enumerate(genres):
            genre_target = base_genre + (1 if g_idx < genre_remainder else 0)
            short_n = round(genre_target * length_pct["short"])
            medium_n = round(genre_target * length_pct["medium"])
            long_n = genre_target - short_n - medium_n
            for lb, count in {"short": short_n, "medium": medium_n, "long": long_n}.items():
                remaining = count
                while remaining > 0:
                    n = min(chunk_size, remaining)
                    jobs.append({"job_id": job_counter, "topic": topic, "genre": genre, "length_bucket": lb, "n": n})
                    job_counter += 1
                    remaining -= n
    return jobs

banking_batch_plan = build_banking_batch_plan(total_target=3000, chunk_size=25)
print("Banking batches planned:", len(banking_batch_plan))
print("Banking texts targeted:", sum(j["n"] for j in banking_batch_plan))

# --- RUN THIS FIRST TOMORROW (smoke test, 2 small batches) ---
smoke_1 = run_banking_production_batch(job_id=9001, topic="Debit and credit cards", genre="question", length_bucket="short", n=25)
smoke_2 = run_banking_production_batch(job_id=9002, topic="Banking-app experience", genre="comment", length_bucket="long", n=25)
print("Smoke 1:", smoke_1["passed"], "/", smoke_1["requested"])
print("Smoke 2:", smoke_2["passed"], "/", smoke_2["requested"])

Banking batches planned: 144
Banking texts targeted: 3000
Rows parsed from response: 25
Rows parsed from response: 25
Smoke 1: 25 / 25
Smoke 2: 21 / 25


In [ ]:
failed_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "*_FAILED"))
failed_job_ids = [int(os.path.basename(f).split("_")[2]) for f in failed_folders]
retry_jobs = [job for job in batch_plan if job["job_id"] in failed_job_ids]
print("General jobs to retry tomorrow:", len(retry_jobs))

General jobs to retry tomorrow: 11


In [ ]:
import pandas as pd

rejected_path = os.path.join(
    SYNTHETIC_ROOT, "03_banking_raw_batches", "banking_batch_9002", "rejected_rows.csv"
)

rejected_df = pd.read_csv(rejected_path)
print("Number of rejected rows:", len(rejected_df))
print()
for i, row in rejected_df.iterrows():
    print(f"--- Rejected row {i} ---")
    print("Text:", row["text"])
    print("Reason:", row["notes"])
    print()

Number of rejected rows: 4

--- Rejected row 0 ---
Text: غريبة توني مجرب أدخل الحساب وطلع لي خادم الصيانة مع إنه مو وقت صيانة معتاد حسب ما أعرف
Reason: nan

--- Rejected row 1 ---
Text: يا ليت تحلون مشكلة خروج الحساب التلقائي لأنها مزعجة وتصير معي حتى وأنا قاعد أستخدم التطبيق
Reason: nan

--- Rejected row 2 ---
Text: صفحة تفاصيل الحساب ما صارت تفتح معي مدري المشكلة من جهازي ولا النظام فيه خلل مؤقت
Reason: nan

--- Rejected row 3 ---
Text: البرنامج صار يعلق عندي إذا جيت أبي أحمل كشف الحساب بصيغة معينة ويقفل البرنامج من حاله
Reason: nan



In [ ]:
print("Columns in rejected_rows.csv:", list(rejected_df.columns))
print()

for i, row in rejected_df.iterrows():
    print(f"--- Row {i} (all fields) ---")
    print(row.to_dict())
    print()

Columns in rejected_rows.csv: ['id', 'text', 'pool', 'topic', 'genre', 'length_bucket', 'generator', 'generator_model_or_version', 'prompt_version', 'batch_id', 'created_at', 'audit_status', 'cleaning_status', 'notes', 'reject_reason']

--- Row 0 (all fields) ---
{'id': 'banking_prod_9002_20260830_010', 'text': 'غريبة توني مجرب أدخل الحساب وطلع لي خادم الصيانة مع إنه مو وقت صيانة معتاد حسب ما أعرف', 'pool': 'banking', 'topic': 'Banking-app experience', 'genre': 'comment', 'length_bucket': 'long', 'generator': 'gemini', 'generator_model_or_version': 'gemini-3.5-flash-lite', 'prompt_version': 'v1', 'batch_id': 'banking_prod_9002_20260830', 'created_at': '2026-08-30', 'audit_status': 'pending', 'cleaning_status': 'pending', 'notes': nan, 'reject_reason': 'named_bank_or_product_leakage'}

--- Row 1 (all fields) ---
{'id': 'banking_prod_9002_20260830_014', 'text': 'يا ليت تحلون مشكلة خروج الحساب التلقائي لأنها مزعجة وتصير معي حتى وأنا قاعد أستخدم التطبيق', 'pool': 'banking', 'topic': 'Banki

In [ ]:
print("BANKING_NAMED_TERMS:", BANKING_NAMED_TERMS)
print()

sample_texts = [
    "غريبة توني مجرب أدخل الحساب وطلع لي خادم الصيانة مع إنه مو وقت صيانة معتاد حسب ما أعرف",
    "يا ليت تحلون مشكلة خروج الحساب التلقائي لأنها مزعجة وتصير معي حتى وأنا قاعد أستخدم التطبيق",
    "صفحة تفاصيل الحساب ما صارت تفتح معي مدري المشكلة من جهازي ولا النظام فيه خلل مؤقت",
    "البرنامج صار يعلق عندي إذا جيت أبي أحمل كشف الحساب بصيغة معينة ويقفل البرنامج من حاله",
]

for text in sample_texts:
    matched = [term for term in BANKING_NAMED_TERMS if term in text]
    print("Text:", text)
    print("Matched terms:", matched)
    print()

BANKING_NAMED_TERMS: ['الراجحي', 'الأهلي', 'سامبا', 'الإنماء', 'ساب', 'بنك ساب', 'بنك الرياض', 'بنك الجزيرة', 'بنك البلاد', 'الاستثمار', 'الرياض المالية', 'فيزا', 'ماستر كارد', 'مدى', 'ابل باي', 'Apple Pay', 'STC Pay', 'PayPal', 'بايبال', 'Visa', 'Mastercard']

Text: غريبة توني مجرب أدخل الحساب وطلع لي خادم الصيانة مع إنه مو وقت صيانة معتاد حسب ما أعرف
Matched terms: ['ساب']

Text: يا ليت تحلون مشكلة خروج الحساب التلقائي لأنها مزعجة وتصير معي حتى وأنا قاعد أستخدم التطبيق
Matched terms: ['ساب']

Text: صفحة تفاصيل الحساب ما صارت تفتح معي مدري المشكلة من جهازي ولا النظام فيه خلل مؤقت
Matched terms: ['ساب']

Text: البرنامج صار يعلق عندي إذا جيت أبي أحمل كشف الحساب بصيغة معينة ويقفل البرنامج من حاله
Matched terms: ['ساب']



In [ ]:
import inspect
print(inspect.getsource(validate_banking_rows))

def validate_banking_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]
    for row in local_rows:
        text = row["text"]
        hard_reasons = []
        if not text: hard_reasons.append("blank_text")
        if text in seen_texts: hard_reasons.append("exact_duplicate")
        if any(term.lower() in text.lower() for term in BANKING_NAMED_TERMS): hard_reasons.append("named_bank_or_product_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text): hard_reasons.append("possible_pii")
        if hard_reasons:
            row_copy = dict(row); row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy); continue
        row = dict(row)
        word_count = len(text.split())
        if not (min_words <= word_count <= max_words):
            row["notes"] = f"length_bucket_flag

In [ ]:
import re

# Fix: bare "ساب" (3 letters) was matching *inside* the very common word
# "الحساب" ("account") because the old check used plain substring matching.
# This patch requires a whole-word match instead, so "ساب" still catches real
# mentions of "SAB Bank" but not "الحساب".
#
# Also: "مدى" is both the debit-card network name AND an ordinary Arabic word
# meaning "extent/range" -- spelled identically, so code cannot tell them apart.
# We move it from a hard reject to a soft flag (row is kept, but flagged in
# "notes" for manual audit) instead of silently discarding possibly-valid rows.

BANKING_NAMED_TERMS = [term for term in BANKING_NAMED_TERMS if term != "مدى"]
BANKING_SOFT_FLAG_TERMS = ["مدى"]

def term_hits(text, terms):
    hits = []
    text_lower = text.lower()
    for term in terms:
        pattern = r'\b' + re.escape(term.lower()) + r'\b'
        if re.search(pattern, text_lower, flags=re.UNICODE):
            hits.append(term)
    return hits

def validate_banking_rows(local_rows, length_bucket):
    seen_texts = set()
    passed, rejected = [], []
    length_ranges = {"short": (5, 9), "medium": (10, 20), "long": (21, 30)}
    min_words, max_words = length_ranges[length_bucket]
    for row in local_rows:
        text = row["text"]
        hard_reasons = []
        if not text: hard_reasons.append("blank_text")
        if text in seen_texts: hard_reasons.append("exact_duplicate")
        if term_hits(text, BANKING_NAMED_TERMS): hard_reasons.append("named_bank_or_product_leakage")
        if PHONE_PATTERN.search(text) or ID_PATTERN.search(text): hard_reasons.append("possible_pii")
        if hard_reasons:
            row_copy = dict(row); row_copy["reject_reason"] = ";".join(hard_reasons)
            rejected.append(row_copy); continue
        row = dict(row)
        word_count = len(text.split())
        notes_parts = []
        if not (min_words <= word_count <= max_words):
            notes_parts.append(f"length_bucket_flag (actual words: {word_count})")
        soft_hits = term_hits(text, BANKING_SOFT_FLAG_TERMS)
        if soft_hits:
            notes_parts.append(f"ambiguous_term_flag: {', '.join(soft_hits)}")
        if notes_parts:
            row["notes"] = " | ".join(notes_parts)
        seen_texts.add(text); passed.append(row)
    return passed, rejected

# quick proof the fix works on the exact 4 texts that were wrongly rejected
test_texts = [
    "غريبة توني مجرب أدخل الحساب وطلع لي خادم الصيانة مع إنه مو وقت صيانة معتاد حسب ما أعرف",
    "يا ليت تحلون مشكلة خروج الحساب التلقائي لأنها مزعجة وتصير معي حتى وأنا قاعد أستخدم التطبيق",
    "صفحة تفاصيل الحساب ما صارت تفتح معي مدري المشكلة من جهازي ولا النظام فيه خلل مؤقت",
    "البرنامج صار يعلق عندي إذا جيت أبي أحمل كشف الحساب بصيغة معينة ويقفل البرنامج من حاله",
]
for t in test_texts:
    print(term_hits(t, BANKING_NAMED_TERMS), "->", t[:40])

[] -> غريبة توني مجرب أدخل الحساب وطلع لي خادم
[] -> يا ليت تحلون مشكلة خروج الحساب التلقائي 
[] -> صفحة تفاصيل الحساب ما صارت تفتح معي مدري
[] -> البرنامج صار يعلق عندي إذا جيت أبي أحمل 


In [ ]:
failed_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "*_FAILED"))
failed_job_ids = [int(os.path.basename(f).split("_")[2]) for f in failed_folders]
retry_jobs = [job for job in batch_plan if job["job_id"] in failed_job_ids]
print("Jobs to retry:", len(retry_jobs))

for job in retry_jobs:
    result = run_general_production_batch(
        job_id=job["job_id"], topic=job["topic"], genre=job["genre"],
        length_bucket=job["length_bucket"], n=job["n"],
    )
    print(job["job_id"], "->", result["passed"], "/", result["requested"])
    time.sleep(1.5)

Jobs to retry: 11
Rows parsed from response: 25
494 -> 25 / 25
Rows parsed from response: 17
495 -> 17 / 17
Job 496 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
496 -> 25 / 25
Rows parsed from response: 25
497 -> 25 / 25
Rows parsed from response: 25
498 -> 25 / 25
Rows parsed from response: 25
499 -> 25 / 25
Rows parsed from response: 25
500 -> 25 / 25
Rows parsed from response: 25
501 -> 25 / 25
Rows parsed from response: 16
502 -> 16 / 16
Job 503 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
503 -> 25 / 25
Rows parsed from response: 25
504 -> 25 / 25


In [ ]:
import glob
import csv

batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "general_batch_*"))
batch_folders = [f for f in batch_folders if not f.endswith("_FAILED")]

all_general_rows = []
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_general_rows.append(row)

print("Total batch folders found:", len(batch_folders))
print("Total rows collected from all batch folders:", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
duplicate_log = []

for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        duplicate_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("Cross-batch exact duplicates removed:", len(duplicate_log))
print("Remaining clean unique rows:", len(deduped_rows))

dedup_log_path = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_cross_batch_dedup_log.csv")
if duplicate_log:
    with open(dedup_log_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=duplicate_log[0].keys())
        writer.writeheader()
        writer.writerows(duplicate_log)

print("Deletion log saved:", dedup_log_path)

Total batch folders found: 504
Total rows collected from all batch folders: 11796
Cross-batch exact duplicates removed: 42
Remaining clean unique rows: 11754
Deletion log saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/general_cross_batch_dedup_log.csv


In [ ]:
shortfall = 12000 - len(deduped_rows)
extra_batches_needed = -(-shortfall // 25)  # ceil division
print("Shortfall:", shortfall, "-> extra batches needed:", extra_batches_needed)

topup_jobs = []
for i in range(extra_batches_needed):
    template = batch_plan[i % len(batch_plan)]
    topup_jobs.append({
        "job_id": 6000 + i,
        "topic": template["topic"],
        "genre": template["genre"],
        "length_bucket": template["length_bucket"],
        "n": 25,
    })

for job in topup_jobs:
    result = run_general_production_batch(
        job_id=job["job_id"], topic=job["topic"], genre=job["genre"],
        length_bucket=job["length_bucket"], n=job["n"],
    )
    print(job["job_id"], "->", result["passed"], "/", result["requested"])
    time.sleep(1.5)

Shortfall: 246 -> extra batches needed: 10
Job 6000 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Job 6000 failed (attempt 2): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
6000 -> 25 / 25
Rows parsed from response: 25
6001 -> 25 / 25
Rows parsed from response: 25
6002 -> 25 / 25
Rows parsed from response: 25
6003 -> 25 / 25
Rows parsed from response: 25
6004 -> 25 / 25
Rows parsed from response: 25
6005 -> 25 / 25
Rows parsed from response: 25
6006 -> 25 / 25
Rows parsed from response: 25
6007 -> 25 / 25
Rows parsed from response: 25
6008 -> 25 / 25
Rows parsed from response: 25
6009 -> 25 / 25


In [ ]:
import glob
import csv
import hashlib

# Step 1: re-collect everything (including the 10 new top-up batches)
batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "general_batch_*"))
batch_folders = [f for f in batch_folders if not f.endswith("_FAILED")]

all_general_rows = []
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_general_rows.append(row)

print("Total batch folders found:", len(batch_folders))
print("Total rows collected:", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
duplicate_log = []

for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        duplicate_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("Cross-batch exact duplicates removed:", len(duplicate_log))
print("Remaining clean unique rows:", len(deduped_rows))

dedup_log_path = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_cross_batch_dedup_log.csv")
if duplicate_log:
    with open(dedup_log_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=duplicate_log[0].keys())
        writer.writeheader()
        writer.writerows(duplicate_log)
print("Deletion log saved:", dedup_log_path)

# Step 2: build the nested 3k / 6k / 12k subsets using a stable seed-based order
FIXED_SEED = "sarf_general_corpus_v1"

def stable_sort_key(row_id):
    return hashlib.sha256(f"{FIXED_SEED}_{row_id}".encode("utf-8")).hexdigest()

sorted_rows = sorted(deduped_rows, key=lambda row: stable_sort_key(row["id"]))

general_3k = sorted_rows[:3000]
general_6k = sorted_rows[:6000]
general_12k = sorted_rows[:12000]

print("3k available:", len(general_3k))
print("6k available:", len(general_6k))
print("12k available:", len(general_12k), "(target: 12000)")

fieldnames = list(deduped_rows[0].keys())
final_dir = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")

save_csv(general_3k, os.path.join(final_dir, "general_3k_draft.csv"), fieldnames)
save_csv(general_6k, os.path.join(final_dir, "general_6k_draft.csv"), fieldnames)
save_csv(general_12k, os.path.join(final_dir, "general_12k_draft.csv"), fieldnames)

print("Saved to:", final_dir)
print(sorted(os.listdir(final_dir)))

Total batch folders found: 514
Total rows collected: 12046
Cross-batch exact duplicates removed: 43
Remaining clean unique rows: 12003
Deletion log saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/general_cross_batch_dedup_log.csv


KeyError: 'id'

In [ ]:
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            if fieldnames is None or "id" not in fieldnames:
                print("Problem file:", passed_path)
                print("Columns found:", fieldnames)
                print()

Problem file: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/02_general_raw_batches/general_batch_0001/passed_rows.csv
Columns found: ['\ufeffid', 'text', 'pool', 'topic', 'genre', 'length_bucket', 'generator', 'generator_model_or_version', 'prompt_version', 'batch_id', 'created_at', 'audit_status', 'cleaning_status', 'notes']

Problem file: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/02_general_raw_batches/general_batch_0002/passed_rows.csv
Columns found: ['\ufeffid', 'text', 'pool', 'topic', 'genre', 'length_bucket', 'generator', 'generator_model_or_version', 'prompt_version', 'batch_id', 'created_at', 'audit_status', 'cleaning_status', 'notes']

Problem file: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/02_general_raw_batches/general_batch_0003/passed_rows.csv
Columns found: ['\ufeffid', 'text', 'pool', 'topic', 'genre', 'length_bucket', 'generator', 'generator_model_or_version', 'prompt_version', 'batch_id', 'created_at'

In [ ]:
import glob
import csv
import hashlib

# Step 1: re-collect everything (encoding="utf-8-sig" ignores the hidden BOM character)
batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "general_batch_*"))
batch_folders = [f for f in batch_folders if not f.endswith("_FAILED")]

all_general_rows = []
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_general_rows.append(row)

print("Total batch folders found:", len(batch_folders))
print("Total rows collected:", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
duplicate_log = []

for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        duplicate_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("Cross-batch exact duplicates removed:", len(duplicate_log))
print("Remaining clean unique rows:", len(deduped_rows))

dedup_log_path = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_cross_batch_dedup_log.csv")
if duplicate_log:
    with open(dedup_log_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=duplicate_log[0].keys())
        writer.writeheader()
        writer.writerows(duplicate_log)
print("Deletion log saved:", dedup_log_path)

# Step 2: build the nested 3k / 6k / 12k subsets using a stable seed-based order
FIXED_SEED = "sarf_general_corpus_v1"

def stable_sort_key(row_id):
    return hashlib.sha256(f"{FIXED_SEED}_{row_id}".encode("utf-8")).hexdigest()

sorted_rows = sorted(deduped_rows, key=lambda row: stable_sort_key(row["id"]))

general_3k = sorted_rows[:3000]
general_6k = sorted_rows[:6000]
general_12k = sorted_rows[:12000]

print("3k available:", len(general_3k))
print("6k available:", len(general_6k))
print("12k available:", len(general_12k), "(target: 12000)")

fieldnames = list(deduped_rows[0].keys())
final_dir = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")

save_csv(general_3k, os.path.join(final_dir, "general_3k_draft.csv"), fieldnames)
save_csv(general_6k, os.path.join(final_dir, "general_6k_draft.csv"), fieldnames)
save_csv(general_12k, os.path.join(final_dir, "general_12k_draft.csv"), fieldnames)

print("Saved to:", final_dir)
print(sorted(os.listdir(final_dir)))

Total batch folders found: 514
Total rows collected: 12046
Cross-batch exact duplicates removed: 43
Remaining clean unique rows: 12003
Deletion log saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/general_cross_batch_dedup_log.csv
3k available: 3000
6k available: 6000
12k available: 12000 (target: 12000)
Saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/05_final_corpora
['general_12k_draft.csv', 'general_3k_draft.csv', 'general_6k_draft.csv']


In [ ]:
total_requested = 0
total_passed = 0

for job in banking_batch_plan:
    result = run_banking_production_batch(
        job_id=job["job_id"], topic=job["topic"], genre=job["genre"],
        length_bucket=job["length_bucket"], n=job["n"],
    )
    total_requested += result["requested"]
    total_passed += result["passed"]
    print(job["job_id"], "->", result["passed"], "/", result["requested"])
    time.sleep(1.5)

print("=== BANKING PRODUCTION DONE ===")
print("Total requested:", total_requested)
print("Total passed:", total_passed)

Rows parsed from response: 25
1 -> 25 / 25
Job 2 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
2 -> 25 / 25
Rows parsed from response: 8
3 -> 8 / 8
Rows parsed from response: 25
4 -> 25 / 25
Rows parsed from response: 25
5 -> 25 / 25
Rows parsed from response: 25
6 -> 25 / 25
Rows parsed from response: 9
7 -> 9 / 9
Rows parsed from response: 25
8 -> 25 / 25
Rows parsed from response: 25
9 -> 25 / 25
Rows parsed from response: 25
10 -> 25 / 25
Rows parsed from response: 8
11 -> 8 / 8
Rows parsed from response: 25
12 -> 25 / 25
Job 13 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
13 -> 25 / 25
Row

In [ ]:
failed_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", "*_FAILED"))
failed_job_ids = [int(os.path.basename(f).split("_")[2]) for f in failed_folders]
retry_jobs = [job for job in banking_batch_plan if job["job_id"] in failed_job_ids]
print("Banking jobs to retry:", len(retry_jobs))

for job in retry_jobs:
    result = run_banking_production_batch(
        job_id=job["job_id"], topic=job["topic"], genre=job["genre"],
        length_bucket=job["length_bucket"], n=job["n"],
    )
    print(job["job_id"], "->", result["passed"], "/", result["requested"])
    time.sleep(1.5)

Banking jobs to retry: 2
Rows parsed from response: 25
33 -> 25 / 25
Rows parsed from response: 25
64 -> 25 / 25


In [ ]:
import glob
import csv
import hashlib

banking_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "03_banking_raw_batches", "banking_batch_*"))
banking_folders = [f for f in banking_folders if not f.endswith("_FAILED")]

all_banking_rows = []
for folder in banking_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_banking_rows.append(row)

print("Total banking batch folders found:", len(banking_folders))
print("Total rows collected:", len(all_banking_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_banking_rows = []
duplicate_log = []

for row in all_banking_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        duplicate_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_banking_rows.append(row)

print("Cross-batch exact duplicates removed:", len(duplicate_log))
print("Remaining clean unique rows:", len(deduped_banking_rows))

dedup_log_path = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "banking_cross_batch_dedup_log.csv")
if duplicate_log:
    with open(dedup_log_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=duplicate_log[0].keys())
        writer.writeheader()
        writer.writerows(duplicate_log)
print("Deletion log saved:", dedup_log_path)

# Same reproducible seed-based selection method used for general
BANKING_SEED = "sarf_banking_corpus_v1"

def banking_sort_key(row_id):
    return hashlib.sha256(f"{BANKING_SEED}_{row_id}".encode("utf-8")).hexdigest()

sorted_banking_rows = sorted(deduped_banking_rows, key=lambda row: banking_sort_key(row["id"]))
banking_3k = sorted_banking_rows[:3000]

print("3k available:", len(banking_3k), "(target: 3000)")

fieldnames = list(deduped_banking_rows[0].keys())
final_dir = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
save_csv(banking_3k, os.path.join(final_dir, "banking_3k_draft.csv"), fieldnames)

print("Saved to:", final_dir)
print(sorted(os.listdir(final_dir)))

Total banking batch folders found: 146
Total rows collected: 3046
Cross-batch exact duplicates removed: 10
Remaining clean unique rows: 3036
Deletion log saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/banking_cross_batch_dedup_log.csv
3k available: 3000 (target: 3000)
Saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/05_final_corpora
['banking_3k_draft.csv', 'general_12k_draft.csv', 'general_3k_draft.csv', 'general_6k_draft.csv']


---
# Part 2: Cleaning, Leakage Check, Manual Audit, and Freeze

**Status:** General (12,000) and Banking (3,000) generation is complete and saved.
This section covers: Saudi frozen test leakage check, stratified manual audit,
final manifest with hashes, and corpus freeze.

## Step 1: Saudi Frozen Test Leakage Check
Checking for exact/normalized text overlap between our synthetic corpus and
`saudi_test_frozen_v1.csv`. This is a protection check only — we never read
the test content itself, only count matches.

In [ ]:
import glob

search_pattern = os.path.join(PROJECT_ROOT, "**", "saudi_test_frozen_v1.csv")
matches = glob.glob(search_pattern, recursive=True)
print("Found:", matches)

Found: ['/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_processed_data/saudi_test_frozen_v1.csv']


In [ ]:
import pandas as pd

saudi_path = matches[0]
saudi_df = pd.read_csv(saudi_path)
print("Columns:", list(saudi_df.columns))
print("Row count:", len(saudi_df))

Columns: ['label', 'text']
Row count: 3580


In [ ]:
import csv

def normalize_for_matching(text):
    return " ".join(text.strip().split())

saudi_texts_exact = set(saudi_df["text"].astype(str))
saudi_texts_normalized = set(normalize_for_matching(t) for t in saudi_df["text"].astype(str))

def check_leakage(csv_path, pool_name):
    exact_matches = 0
    normalized_only_matches = 0
    total_rows = 0
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_rows += 1
            text = row["text"]
            norm = normalize_for_matching(text)
            if text in saudi_texts_exact:
                exact_matches += 1
            elif norm in saudi_texts_normalized:
                normalized_only_matches += 1
    return {
        "pool": pool_name,
        "total_rows": total_rows,
        "exact_matches": exact_matches,
        "normalized_only_matches": normalized_only_matches,
    }

final_dir = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
general_result = check_leakage(os.path.join(final_dir, "general_12k_draft.csv"), "general")
banking_result = check_leakage(os.path.join(final_dir, "banking_3k_draft.csv"), "banking")

print(general_result)
print(banking_result)

audit_dir = os.path.join(SYNTHETIC_ROOT, "06_final_audit")
os.makedirs(audit_dir, exist_ok=True)
report_path = os.path.join(audit_dir, "saudi_test_leakage_report.csv")
with open(report_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["pool", "total_rows", "exact_matches", "normalized_only_matches"])
    writer.writeheader()
    writer.writerow(general_result)
    writer.writerow(banking_result)

print("Leakage report saved:", report_path)

{'pool': 'general', 'total_rows': 12000, 'exact_matches': 0, 'normalized_only_matches': 0}
{'pool': 'banking', 'total_rows': 3000, 'exact_matches': 0, 'normalized_only_matches': 0}
Leakage report saved: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/saudi_test_leakage_report.csv


In [ ]:
import random
import csv
from collections import defaultdict

AUDIT_SEED = 42

def load_rows(csv_path):
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        return list(reader)

def stratified_sample(rows, target_size, seed):
    groups = defaultdict(list)
    for row in rows:
        key = (row["topic"], row["genre"], row["length_bucket"])
        groups[key].append(row)

    total = len(rows)
    rng = random.Random(seed)
    sample = []
    for key, group_rows in groups.items():
        group_share = len(group_rows) / total
        group_target = max(1, round(group_share * target_size))
        group_target = min(group_target, len(group_rows))
        sample.extend(rng.sample(group_rows, group_target))

    rng.shuffle(sample)
    return sample[:target_size]

final_dir = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
general_rows = load_rows(os.path.join(final_dir, "general_12k_draft.csv"))
banking_rows = load_rows(os.path.join(final_dir, "banking_3k_draft.csv"))

general_sample = stratified_sample(general_rows, 300, seed=AUDIT_SEED)
banking_sample = stratified_sample(banking_rows, 300, seed=AUDIT_SEED + 1)

print("General audit sample size:", len(general_sample))
print("Banking audit sample size:", len(banking_sample))

audit_dir = os.path.join(SYNTHETIC_ROOT, "06_final_audit")
os.makedirs(audit_dir, exist_ok=True)

def save_audit_sample(sample, filepath):
    fieldnames = ["id", "text", "pool", "topic", "genre", "length_bucket", "audit_result", "audit_notes"]
    with open(filepath, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in sample:
            writer.writerow({
                "id": row["id"], "text": row["text"], "pool": row["pool"],
                "topic": row["topic"], "genre": row["genre"], "length_bucket": row["length_bucket"],
                "audit_result": "", "audit_notes": "",
            })

save_audit_sample(general_sample, os.path.join(audit_dir, "general_audit_sample.csv"))
save_audit_sample(banking_sample, os.path.join(audit_dir, "banking_audit_sample.csv"))
print("Audit sample files saved to:", audit_dir)

General audit sample size: 288
Banking audit sample size: 291
Audit sample files saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit


In [ ]:
def stratified_sample(rows, target_size, seed):
    groups = defaultdict(list)
    for row in rows:
        key = (row["topic"], row["genre"], row["length_bucket"])
        groups[key].append(row)

    total = len(rows)
    rng = random.Random(seed)
    sample = []
    selected_ids = set()
    for key, group_rows in groups.items():
        group_share = len(group_rows) / total
        group_target = max(1, round(group_share * target_size))
        group_target = min(group_target, len(group_rows))
        chosen = rng.sample(group_rows, group_target)
        for row in chosen:
            if row["id"] not in selected_ids:
                sample.append(row)
                selected_ids.add(row["id"])

    # top up to hit target_size exactly if rounding left us short
    if len(sample) < target_size:
        remaining = [row for row in rows if row["id"] not in selected_ids]
        rng.shuffle(remaining)
        needed = target_size - len(sample)
        sample.extend(remaining[:needed])

    rng.shuffle(sample)
    return sample[:target_size]

general_sample = stratified_sample(general_rows, 300, seed=AUDIT_SEED)
banking_sample = stratified_sample(banking_rows, 300, seed=AUDIT_SEED + 1)

print("General audit sample size:", len(general_sample))
print("Banking audit sample size:", len(banking_sample))

save_audit_sample(general_sample, os.path.join(audit_dir, "general_audit_sample.csv"))
save_audit_sample(banking_sample, os.path.join(audit_dir, "banking_audit_sample.csv"))
print("Audit sample files re-saved to:", audit_dir)

General audit sample size: 300
Banking audit sample size: 300
Audit sample files re-saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit


In [ ]:
import csv

# هذي الدالة تفتح ملف عينة المراجعة (general أو banking) وتعرض كل نص
# لك عشان تحكمي عليه يدويًا: Pass / Fail / Unsure
# القرار ينحفظ بالملف مباشرة بعد كل نص، عشان لو صار انقطاع ما نخسر شي
def run_manual_audit(sample_path):
    # نقرأ كل الصفوف من ملف العينة
    with open(sample_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    fieldnames = list(rows[0].keys())

    # نحدد الصفوف اللي لسه ما انحكم عليها (audit_result فاضي)
    # هذا اللي يخلي الأداة "تكمل من وين وقفت" لو رجعتي تشغلينها
    remaining = [r for r in rows if r["audit_result"].strip() == ""]
    already_done = len(rows) - len(remaining)
    print(f"Total rows: {len(rows)} | Already audited: {already_done} | Remaining: {len(remaining)}")
    print("Type: p = Pass, f = Fail, u = Unsure, q = stop and save")
    print("-" * 50)

    for row in remaining:
        # نعرض النص والمعلومات المرافقة له (الموضوع، النوع، الطول)
        print(f"\nID: {row['id']} | Topic: {row['topic']} | Genre: {row['genre']} | Length: {row['length_bucket']}")
        print(f"Text: {row['text']}")

        decision = input("Your decision (p/f/u/q): ").strip().lower()

        if decision == "q":
            print("Stopping. Progress is saved.")
            break

        # نحول اختصار القرار إلى القيمة الكاملة المطلوبة بالملف
        if decision == "p":
            row["audit_result"] = "Pass"
        elif decision == "f":
            row["audit_result"] = "Fail"
        elif decision == "u":
            row["audit_result"] = "Unsure"
        else:
            print("Invalid input, this row stays unaudited, we'll ask again next run.")
            continue

        # ملاحظة اختيارية بس لو تبين توضحين سبب القرار
        note = input("Optional note (press Enter to skip): ").strip()
        row["audit_notes"] = note

        # نحفظ الملف كامل بعد كل قرار مباشرة، عشان أي تقدم ما يضيع
        with open(sample_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

    print("\nSession ended.")

In [ ]:
# نبدأ بمراجعة عينة general (300 نص)
run_manual_audit(os.path.join(audit_dir, "general_audit_sample.csv"))

Total rows: 300 | Already audited: 19 | Remaining: 281
Type: p = Pass, f = Fail, u = Unsure, q = stop and save
--------------------------------------------------

ID: general_prod_0329_20260829_016 | Topic: Technology and devices | Genre: comment | Length: medium
Text: التلفزيون الجديد ألوانه رهيبة وتخلي مشاهدة الأفلام بالبيت تشبه السينما.
Your decision (p/f/u/q): p
Optional note (press Enter to skip): 

ID: general_prod_0172_20260829_008 | Topic: Sports and fitness | Genre: question | Length: short
Text: كم سعرة حرارية أحرق في الساعة؟
Your decision (p/f/u/q): p
Optional note (press Enter to skip): 

ID: general_prod_0116_20260829_021 | Topic: Study and university life | Genre: comment | Length: short
Text: المعمل حق الكيمياء مرتب ونظيف ومجهز بالكامل.
Your decision (p/f/u/q): p
Optional note (press Enter to skip): 

ID: general_prod_0386_20260829_018 | Topic: Family and social life | Genre: question | Length: medium
Text: مين يبي يروح معنا للسوق نشتري لوازم الطلعة البرية حق الصباح؟
You

In [ ]:
# Shrink banking audit sample to 60 rows (general alone already meets the 300 minimum)
banking_sample_small = stratified_sample(banking_rows, 60, seed=AUDIT_SEED + 1)

save_audit_sample(banking_sample_small, os.path.join(audit_dir, "banking_audit_sample.csv"))
print("Banking audit sample reduced to:", len(banking_sample_small))

Banking audit sample reduced to: 60


In [ ]:
import csv

sample_path = os.path.join(audit_dir, "general_audit_sample.csv")
with open(sample_path, "r", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

already_audited = [r for r in rows if r["audit_result"].strip() != ""]
print(f"Rows already judged: {len(already_audited)}")
for r in already_audited:
    print(f"{r['id']} -> {r['audit_result']} | {r['audit_notes']} | {r['text'][:70]}")

Rows already judged: 25
general_prod_0329_20260829_002 -> Pass |  | الكاميرا حقت الجوال الجديد تعطي صور بالليل تفتح النفس وواضحة مرة.
general_prod_0037_20260829_025 -> Pass |  | البطاطس المقلي كان بارد وناشف شوي وما قدرت أكمله للأسف الشديد.
general_prod_0026_20260829_007 -> Pass |  | جربت الفطائر التركية بالجبنة السائلة وحسيت إن العجينة هشة وناشفة شوي.
general_prod_0159_20260829_015 -> Pass |  | ساعات العمل اليوم مرت بسرعة ما حسيت فيها.
general_prod_0052_20260829_005 -> Pass |  | شرايكم يا جماعة، السفر لـ ماليزيا بالأسرة زين ولا المتعة للشباب أكثر؟
general_prod_0451_20260829_005 -> Fail | too MSA-like / lacks Saudi dialect markers | الوصول للجمعية التعاونية أخذ مني وقت طويل.
general_prod_0338_20260829_010 -> Pass |  | وش أحسن ماركة مكيفات هواء بالسوق حالياً؟
general_prod_0019_20260829_006 -> Pass |  | الخبز الحار من الفرن طعمه رهيب.
general_prod_0274_20260829_017 -> Pass |  | تطبيق الخرائط أنقذني اليوم ووجهني لطريق فاضي وبعيد عن الزحمة.
general_prod_0147_20260829_023 -> Pass |  | الجلس

In [ ]:
ids_to_fix = {
    "general_prod_0451_20260829_005": "Pass",
    "general_prod_0498_20260830_024": "Pass",
}

for r in rows:
    if r["id"] in ids_to_fix:
        r["audit_result"] = ids_to_fix[r["id"]]
        r["audit_notes"] = "recalibrated: minor MSA word / expressive variation is not disqualifying alone"

with open(sample_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Updated:", list(ids_to_fix.keys()))

Updated: ['general_prod_0451_20260829_005', 'general_prod_0498_20260830_024']


In [ ]:
import csv

REASON_CATEGORIES = ["language", "logic", "genre", "banking_leakage", "PII", "duplicate", "other"]

def run_manual_audit(sample_path):
    with open(sample_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    fieldnames = list(rows[0].keys())

    remaining = [r for r in rows if r["audit_result"].strip() == ""]
    already_done = len(rows) - len(remaining)
    print(f"Total rows: {len(rows)} | Already audited: {already_done} | Remaining: {len(remaining)}")
    print("Type: p = Pass, f = Fail, u = Unsure, q = stop and save")
    print("Reason categories (for Fail/Unsure only):", ", ".join(REASON_CATEGORIES))
    print("-" * 50)

    for row in remaining:
        print(f"\nID: {row['id']} | Topic: {row['topic']} | Genre: {row['genre']} | Length: {row['length_bucket']}")
        print(f"Text: {row['text']}")

        decision = input("Your decision (p/f/u/q): ").strip().lower()

        if decision == "q":
            print("Stopping. Progress is saved.")
            break

        if decision == "p":
            row["audit_result"] = "Pass"
            row["audit_notes"] = ""
        elif decision in ("f", "u"):
            row["audit_result"] = "Fail" if decision == "f" else "Unsure"
            category = input(f"Reason category ({'/'.join(REASON_CATEGORIES)}): ").strip().lower()
            row["audit_notes"] = category
        else:
            print("Invalid input, this row stays unaudited, we'll ask again next run.")
            continue

        with open(sample_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

    print("\nSession ended.")

In [ ]:
run_manual_audit(os.path.join(audit_dir, "general_audit_sample.csv"))

Total rows: 300 | Already audited: 300 | Remaining: 0
Type: p = Pass, f = Fail, u = Unsure, q = stop and save
Reason categories (for Fail/Unsure only): language, logic, genre, banking_leakage, PII, duplicate, other
--------------------------------------------------

Session ended.


In [ ]:
import os, csv

AUDIT_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "general_audit_sample.csv")

with open(AUDIT_PATH, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print("Columns found:", reader.fieldnames)
print("Total rows:", len(rows))
print()
print("Example row:")
print(rows[0])

Columns found: ['id', 'text', 'pool', 'topic', 'genre', 'length_bucket', 'audit_result', 'audit_notes']
Total rows: 300

Example row:
{'id': 'general_prod_0329_20260829_002', 'text': 'الكاميرا حقت الجوال الجديد تعطي صور بالليل تفتح النفس وواضحة مرة.', 'pool': 'general', 'topic': 'Technology and devices', 'genre': 'comment', 'length_bucket': 'medium', 'audit_result': 'Pass', 'audit_notes': ''}


In [ ]:
from collections import Counter

result_counts = Counter(row["audit_result"].strip() for row in rows)
print("Decision breakdown:")
for k, v in result_counts.items():
    print(f"  {k}: {v}")

fail_unsure_rows = [row for row in rows if row["audit_result"].strip() in ("Fail", "Unsure")]
print()
print("Total Fail/Unsure rows to deep-review:", len(fail_unsure_rows))

# Save them to a separate file for the second review pass
REVIEW_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "general_audit_review_needed.csv")
with open(REVIEW_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(fail_unsure_rows)

print()
print("Saved to:", REVIEW_PATH)
print()
print("Preview of rows to review:")
for row in fail_unsure_rows:
    print(f"- [{row['audit_result']}] {row['id']} | notes: {row['audit_notes']}")

Decision breakdown:
  Pass: 268
  Fail: 30
  Unsure: 2

Total Fail/Unsure rows to deep-review: 32

Saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/general_audit_review_needed.csv

Preview of rows to review:
- [Fail] general_prod_0254_20260829_022 | notes: غير مفهوم
- [Unsure] general_prod_0359_20260829_001 | notes: awkward clause order / run-on structure
- [Fail] general_prod_0237_20260829_019 | notes: language
- [Fail] general_prod_0075_20260829_005 | notes: logic
- [Fail] general_prod_0019_20260829_014 | notes: logic
- [Fail] general_prod_0160_20260829_014 | notes: language
- [Fail] general_prod_0457_20260829_012 | notes: langiage
- [Fail] general_prod_0491_20260829_024 | notes: language/logic
- [Fail] general_prod_0426_20260829_001 | notes: language
- [Fail] general_prod_0226_20260829_007 | notes: other
- [Fail] general_prod_0039_20260829_018 | notes: logic
- [Fail] general_prod_0245_20260829_018 | notes: logic
- [Fail] general_prod_0477_20

In [ ]:
print(f"Total rows to review: {len(fail_unsure_rows)}")
print("=" * 60)
for i, row in enumerate(fail_unsure_rows, 1):
    print(f"{i}. ID: {row['id']}")
    print(f"   Topic: {row['topic']} | Genre: {row['genre']} | Length: {row['length_bucket']}")
    print(f"   Text: {row['text']}")
    print(f"   Current: {row['audit_result']} | Notes: {row['audit_notes']}")
    print("-" * 60)


Total rows to review: 32
1. ID: general_prod_0254_20260829_022
   Topic: Transport and traffic | Genre: question | Length: short
   Text: ليش إشارة المرور معلقة طويلة؟
   Current: Fail | Notes: غير مفهوم
------------------------------------------------------------
2. ID: general_prod_0359_20260829_001
   Topic: Shopping and products (non-financial) | Genre: statement | Length: medium
   Text: والله العروض اللي بالمحل الجديد مرة قوية وتستاهل العنوة اليوم، زحمة عندهم.
   Current: Unsure | Notes: awkward clause order / run-on structure
------------------------------------------------------------
3. ID: general_prod_0237_20260829_019
   Topic: Entertainment and hobbies | Genre: statement | Length: long
   Text: تركيب المكعبات الكبيرة بناء المجسمات المعمارية الشهيرة يشغل ساعات طويلة من الفراغ بأسلوب ممتع ومفيد جداً للعقل،general
   Current: Fail | Notes: language
------------------------------------------------------------
4. ID: general_prod_0075_20260829_005
   Topic: Travel and trips | G

In [ ]:
import os, csv, re

LATIN_PATTERN = re.compile(r'[a-zA-Z]')

def scan_for_latin_contamination(csv_path, pool_name):
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    contaminated = [row for row in rows if LATIN_PATTERN.search(row["text"])]
    print(f"{pool_name}: {len(contaminated)} / {len(rows)} rows contain Latin characters")
    return contaminated

GENERAL_12K_PATH = os.path.join(SYNTHETIC_ROOT, "05_final_corpora", "general_12k_draft.csv")
BANKING_3K_PATH = os.path.join(SYNTHETIC_ROOT, "05_final_corpora", "banking_3k_draft.csv")

general_contaminated = scan_for_latin_contamination(GENERAL_12K_PATH, "General (12,000)")
banking_contaminated = scan_for_latin_contamination(BANKING_3K_PATH, "Banking (3,000)")

print()
print("=== General contaminated rows ===")
for row in general_contaminated:
    print(f"- {row['id']}: {row['text']}")

print()
print("=== Banking contaminated rows ===")
for row in banking_contaminated:
    print(f"- {row['id']}: {row['text']}")

General (12,000): 128 / 12000 rows contain Latin characters
Banking (3,000): 14 / 3000 rows contain Latin characters

=== General contaminated rows ===
- general_prod_0368_20260829_015: الشنطه واسعه وتكhelا اغراض واجد.
- general_prod_0373_20260829_023: الفرشة الرياضية اللي شريتها عشان تمارين البيت مانعة للانزلاق وثابتة بقوة على البلاط، خامتها ممتازة وماتتأثر بالحركة الكثيرة والضغط، أنصح فيها كل مهتم بالرياضة،general
- general_prod_0237_20260829_015: الذهاب في رحلة صيد بحرية مع الربع بالفجرية يحتاج صبر طويل بس نسيم البحر والوناسة تنسيك كل هموم الدنيا،general
- general_prod_0373_20260829_025: السماعة اللاسلكية اللي وصلتني اليوم عزلها للضوضاء قوي وممتاز، الصوت فيها نقي وواضح والتحكم باللمس سهل جداً وما يعلق وقت الرد على المكالمات،general
- general_prod_0237_20260829_025: تسلق الجبال والمرتفعات في عطلة نهاية الأسبوع يعلم الصبر وقوة التحمل ويعطيك شعور بالنصر يوم توصل القمة،general
- general_prod_0373_20260829_017: النبتة الصناعية اللي حطيتها بزاوية الصالة أعطت المكان حياة وبهجة، شكلها كأنها

In [ ]:
import glob, csv, hashlib, re, os

LATIN_PATTERN = re.compile(r'[a-zA-Z]')

batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "general_batch_*"))
batch_folders = [f for f in batch_folders if not f.endswith("_FAILED")]

all_general_rows = []
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_general_rows.append(row)

print("Total raw rows loaded:", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
dup_log = []
for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        dup_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("After dedup:", len(deduped_rows), "| duplicates removed:", len(dup_log))

clean_rows = [row for row in deduped_rows if not LATIN_PATTERN.search(row["text"])]
contaminated_rows = [row for row in deduped_rows if LATIN_PATTERN.search(row["text"])]

print("After removing Latin-contaminated rows:", len(clean_rows), "| removed:", len(contaminated_rows))

CONTAM_LOG_PATH = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_latin_contamination_log.csv")
with open(CONTAM_LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_general_rows[0].keys())
    writer.writeheader()
    writer.writerows(contaminated_rows)
print("Contamination log saved to:", CONTAM_LOG_PATH)

print()
print("Do we have enough clean rows for 12,000?", len(clean_rows) >= 12000)

Total raw rows loaded: 12046
After dedup: 12003 | duplicates removed: 43
After removing Latin-contaminated rows: 11875 | removed: 128
Contamination log saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/04_cleaning_logs/general_latin_contamination_log.csv

Do we have enough clean rows for 12,000? False


In [ ]:
try:
    print("run_general_production_batch exists:", callable(run_general_production_batch))
    print("batch_plan exists, length:", len(batch_plan))
except NameError as e:
    print("MISSING:", e)

run_general_production_batch exists: True
batch_plan exists, length: 504


In [ ]:
import inspect
print(inspect.signature(run_general_production_batch))
print()
print("Example batch_plan entry:")
print(batch_plan[0])

(job_id, topic, genre, length_bucket, n, max_retries=2)

Example batch_plan entry:
{'job_id': 1, 'topic': 'Food and restaurants', 'genre': 'question', 'length_bucket': 'short', 'n': 25}


In [ ]:
NEW_JOB_IDS = list(range(8000, 8008))  # 8 jobs x 25 = 200 raw rows (buffer above the 125 needed)
topup_results = []
for i, job_id in enumerate(NEW_JOB_IDS):
    spec = batch_plan[i % len(batch_plan)]
    result = run_general_production_batch(
        job_id=job_id,
        topic=spec["topic"],
        genre=spec["genre"],
        length_bucket=spec["length_bucket"],
        n=spec["n"],
    )
    topup_results.append(result)
    print(f"Job {job_id}: {result}")

Job 8000 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
Job 8000: {'batch_id': 'general_prod_8000_20260830', 'requested': 25, 'parsed': 25, 'passed': 25, 'rejected': 0, 'rejection_reasons': [], 'failed': False}
Job 8001 failed (attempt 1): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Rows parsed from response: 25
Job 8001: {'batch_id': 'general_prod_8001_20260830', 'requested': 25, 'parsed': 25, 'passed': 25, 'rejected': 0, 'rejection_reasons': [], 'failed': False}
Rows parsed from response: 25
Job 8002: {'batch_id': 'general_prod_8002_20260830', 'requested': 25, 'parsed': 25, 'passed': 25, 'rejected': 0, 'rejection_reasons': [], 'failed': Fals

In [ ]:
import glob, csv, hashlib, re, os

LATIN_PATTERN = re.compile(r'[a-zA-Z]')

batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "02_general_raw_batches", "general_batch_*"))
batch_folders = [f for f in batch_folders if not f.endswith("_FAILED")]

all_general_rows = []
for folder in batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_general_rows.append(row)

print("Total raw rows loaded (including new top-up):", len(all_general_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
deduped_rows = []
dup_log = []
for row in all_general_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        dup_log.append(row)
        continue
    seen_normalized.add(norm)
    deduped_rows.append(row)

print("After dedup:", len(deduped_rows), "| duplicates removed:", len(dup_log))

clean_rows = [row for row in deduped_rows if not LATIN_PATTERN.search(row["text"])]
contaminated_rows = [row for row in deduped_rows if LATIN_PATTERN.search(row["text"])]

print("After removing Latin-contaminated rows:", len(clean_rows), "| removed:", len(contaminated_rows))
print()
print("Do we have enough clean rows for 12,000 now?", len(clean_rows) >= 12000)

Total raw rows loaded (including new top-up): 12238
After dedup: 12193 | duplicates removed: 45
After removing Latin-contaminated rows: 12065 | removed: 128

Do we have enough clean rows for 12,000 now? True


In [ ]:
FIXED_SEED = "sarf_general_corpus_v1"

def stable_sort_key(row_id):
    return hashlib.sha256(f"{FIXED_SEED}_{row_id}".encode("utf-8")).hexdigest()

sorted_rows = sorted(clean_rows, key=lambda row: stable_sort_key(row["id"]))

general_3k = sorted_rows[:3000]
general_6k = sorted_rows[:6000]
general_12k = sorted_rows[:12000]

print("general_3k:", len(general_3k))
print("general_6k:", len(general_6k))
print("general_12k:", len(general_12k))

assert set(r["id"] for r in general_3k) <= set(r["id"] for r in general_6k)
assert set(r["id"] for r in general_6k) <= set(r["id"] for r in general_12k)
print("Nesting check passed.")

def save_csv(rows, filepath, fieldnames):
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

fieldnames = list(all_general_rows[0].keys())
FINAL_DIR = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
save_csv(general_3k, os.path.join(FINAL_DIR, "general_3k_draft.csv"), fieldnames)
save_csv(general_6k, os.path.join(FINAL_DIR, "general_6k_draft.csv"), fieldnames)
save_csv(general_12k, os.path.join(FINAL_DIR, "general_12k_draft.csv"), fieldnames)

DEDUP_LOG_PATH = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_cross_batch_dedup_log.csv")
save_csv(dup_log, DEDUP_LOG_PATH, fieldnames)
CONTAM_LOG_PATH = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "general_latin_contamination_log.csv")
save_csv(contaminated_rows, CONTAM_LOG_PATH, fieldnames)

print()
print("Saved clean final general_3k/6k/12k + updated logs.")

# Check how many of our 300 audited rows survived into the new 12k
new_12k_ids = set(r["id"] for r in general_12k)
audited_ids = set(row["id"] for row in rows)  # from the audit CSV loaded earlier
missing_from_new = audited_ids - new_12k_ids
print()
print("Audited rows no longer in final 12k:", len(missing_from_new))
for mid in missing_from_new:
    print(" -", mid)

general_3k: 3000
general_6k: 6000
general_12k: 12000
Nesting check passed.

Saved clean final general_3k/6k/12k + updated logs.

Audited rows no longer in final 12k: 8
 - general_prod_0224_20260829_011
 - general_prod_0426_20260829_001
 - general_prod_0226_20260829_007
 - general_prod_0307_20260829_001
 - general_prod_0237_20260829_019
 - general_prod_0361_20260829_021
 - general_prod_0303_20260829_010
 - general_prod_0200_20260829_004


In [ ]:
import glob, csv, hashlib, re, os

LATIN_PATTERN = re.compile(r'[a-zA-Z]')

banking_batch_folders = glob.glob(os.path.join(SYNTHETIC_ROOT, "*", "banking_batch_*"))
banking_batch_folders = [f for f in banking_batch_folders if not f.endswith("_FAILED")]
print("Found", len(banking_batch_folders), "banking batch folders")

all_banking_rows = []
for folder in banking_batch_folders:
    passed_path = os.path.join(folder, "passed_rows.csv")
    if os.path.exists(passed_path):
        with open(passed_path, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                all_banking_rows.append(row)

print("Total raw banking rows loaded:", len(all_banking_rows))

def normalize_for_matching(text):
    return " ".join(text.strip().split())

seen_normalized = set()
banking_deduped_rows = []
banking_dup_log = []
for row in all_banking_rows:
    norm = normalize_for_matching(row["text"])
    if norm in seen_normalized:
        banking_dup_log.append(row)
        continue
    seen_normalized.add(norm)
    banking_deduped_rows.append(row)

print("After dedup:", len(banking_deduped_rows), "| duplicates removed:", len(banking_dup_log))

banking_clean_rows = [row for row in banking_deduped_rows if not LATIN_PATTERN.search(row["text"])]
banking_contaminated_rows = [row for row in banking_deduped_rows if LATIN_PATTERN.search(row["text"])]

print("After removing Latin-contaminated rows:", len(banking_clean_rows), "| removed:", len(banking_contaminated_rows))
print()
print("Do we have enough clean rows for 3,000?", len(banking_clean_rows) >= 3000)

Found 146 banking batch folders
Total raw banking rows loaded: 3046
After dedup: 3036 | duplicates removed: 10
After removing Latin-contaminated rows: 3022 | removed: 14

Do we have enough clean rows for 3,000? True


In [ ]:
BANKING_SEED = "sarf_banking_corpus_v1"

def banking_stable_sort_key(row_id):
    return hashlib.sha256(f"{BANKING_SEED}_{row_id}".encode("utf-8")).hexdigest()

banking_sorted_rows = sorted(banking_clean_rows, key=lambda row: banking_stable_sort_key(row["id"]))
banking_3k = banking_sorted_rows[:3000]

print("banking_3k:", len(banking_3k))

def save_csv(rows, filepath, fieldnames):
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

fieldnames = list(all_banking_rows[0].keys())
FINAL_DIR = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
save_csv(banking_3k, os.path.join(FINAL_DIR, "banking_3k_draft.csv"), fieldnames)

DEDUP_LOG_PATH = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "banking_cross_batch_dedup_log.csv")
save_csv(banking_dup_log, DEDUP_LOG_PATH, fieldnames)
CONTAM_LOG_PATH = os.path.join(SYNTHETIC_ROOT, "04_cleaning_logs", "banking_latin_contamination_log.csv")
save_csv(banking_contaminated_rows, CONTAM_LOG_PATH, fieldnames)

print("Saved clean final banking_3k + updated logs.")

banking_3k: 3000
Saved clean final banking_3k + updated logs.


In [ ]:
try:
    print("stratified_sample exists:", callable(stratified_sample))
    print("save_audit_sample exists:", callable(save_audit_sample))
except NameError as e:
    print("MISSING:", e)

stratified_sample exists: True
save_audit_sample exists: True


In [ ]:
new_banking_sample = stratified_sample(banking_3k, 60, 43)
print("New banking audit sample size:", len(new_banking_sample))

BANKING_AUDIT_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "banking_audit_sample.csv")
save_audit_sample(new_banking_sample, BANKING_AUDIT_PATH)

print("Saved to:", BANKING_AUDIT_PATH)

New banking audit sample size: 60
Saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/banking_audit_sample.csv


In [ ]:
try:
    FINAL_DIR = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")
    print("=== Re-checking Saudi test leakage on CLEANED final corpora ===")
    general_leak = check_leakage(os.path.join(FINAL_DIR, "general_12k_draft.csv"), "general")
    print("General result:", general_leak)
    banking_leak = check_leakage(os.path.join(FINAL_DIR, "banking_3k_draft.csv"), "banking")
    print("Banking result:", banking_leak)
except NameError as e:
    print("MISSING:", e)

=== Re-checking Saudi test leakage on CLEANED final corpora ===
General result: {'pool': 'general', 'total_rows': 12000, 'exact_matches': 0, 'normalized_only_matches': 0}
Banking result: {'pool': 'banking', 'total_rows': 3000, 'exact_matches': 0, 'normalized_only_matches': 0}


In [ ]:
report_rows = [general_leak, banking_leak]
REPORT_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "saudi_test_leakage_report.csv")
with open(REPORT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(general_leak.keys()))
    writer.writeheader()
    writer.writerows(report_rows)
print("Updated leakage report saved to:", REPORT_PATH)

Updated leakage report saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/saudi_test_leakage_report.csv


In [ ]:
BANKING_AUDIT_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "banking_audit_sample.csv")
run_manual_audit(BANKING_AUDIT_PATH)

Total rows: 60 | Already audited: 0 | Remaining: 60
Type: p = Pass, f = Fail, u = Unsure, q = stop and save
Reason categories (for Fail/Unsure only): language, logic, genre, banking_leakage, PII, duplicate, other
--------------------------------------------------

ID: banking_prod_0133_20260830_016 | Topic: Cash machines and cash withdrawal | Genre: statement | Length: medium
Text: الصرافة طفت فجأة علي وأنا قاعد أنتظر النقدية تطلع من الفتحة السفلية للجهاز.
Your decision (p/f/u/q): p

ID: banking_prod_0112_20260830_013 | Topic: Payments and merchants | Genre: statement | Length: long
Text: ألغيت الطلب حق السلعة من المتجر بس الفلوس طولت وما نزلت في حسابي البنكي مثل كل مرة تعودت عليها.
Your decision (p/f/u/q): p

ID: banking_prod_0097_20260830_024 | Topic: Payments and merchants | Genre: question | Length: short
Text: وش سبب تعليق عملية الدفع هذه؟
Your decision (p/f/u/q): p

ID: banking_prod_0009_20260830_013 | Topic: Money transfers and transfer status | Genre: statement | Length: short


In [ ]:
from collections import Counter
import os, csv

BANKING_AUDIT_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "banking_audit_sample.csv")

with open(BANKING_AUDIT_PATH, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    banking_rows = list(reader)

result_counts = Counter(row["audit_result"].strip() for row in banking_rows)
print("Total rows:", len(banking_rows))
print("Decision breakdown:")
for k, v in result_counts.items():
    print(f"  {k}: {v}")

banking_fail_unsure = [row for row in banking_rows if row["audit_result"].strip() in ("Fail", "Unsure")]
print()
print("Fail/Unsure rows:")
for row in banking_fail_unsure:
    print(f"- [{row['audit_result']}] {row['id']} | notes: {row['audit_notes']}")

Total rows: 60
Decision breakdown:
  Pass: 60

Fail/Unsure rows:


In [ ]:
import hashlib, json
from datetime import datetime, timezone

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

FINAL_DIR = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")

files_to_hash = {
    "general_3k": "general_3k_draft.csv",
    "general_6k": "general_6k_draft.csv",
    "general_12k": "general_12k_draft.csv",
    "banking_3k": "banking_3k_draft.csv",
}

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generator_model": "gemini-3.5-flash-lite",
    "files": {}
}

for name, filename in files_to_hash.items():
    path = os.path.join(FINAL_DIR, filename)
    with open(path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        row_count = sum(1 for _ in reader)
    manifest["files"][name] = {
        "filename": filename,
        "sha256": file_sha256(path),
        "row_count": row_count,
    }

manifest["quality_audit"] = {
    "general_sample_audited": 300,
    "general_pass": 268,
    "general_fail": 30,
    "general_unsure": 2,
    "banking_sample_audited": 60,
    "banking_pass": 60,
    "banking_fail": 0,
    "banking_unsure": 0,
}

manifest["saudi_test_leakage_check"] = {
    "general": general_leak,
    "banking": banking_leak,
}

MANIFEST_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "corpus_manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifest saved to:", MANIFEST_PATH)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

Manifest saved to: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/corpus_manifest.json
{
  "generated_at_utc": "2026-08-30T16:03:03.987704+00:00",
  "generator_model": "gemini-3.5-flash-lite",
  "files": {
    "general_3k": {
      "filename": "general_3k_draft.csv",
      "sha256": "ad5baedabce3537b9f16c46541a9192a35cc8718bfc1c45c9075ac63a01f608d",
      "row_count": 3000
    },
    "general_6k": {
      "filename": "general_6k_draft.csv",
      "sha256": "8ba68d9dc833842fef642f5ae23fe9ac506ebe778a5da10f7952216e2ce9d092",
      "row_count": 6000
    },
    "general_12k": {
      "filename": "general_12k_draft.csv",
      "sha256": "c8ebca897bc7b4542e1b0420683eb7daf188cffd2944a43842acd52e3e07f49b",
      "row_count": 12000
    },
    "banking_3k": {
      "filename": "banking_3k_draft.csv",
      "sha256": "b2462358d9f3c2e53870f04b691273812ae0be5b8e79b1ce719fc3675f656e9b",
      "row_count": 3000
    }
  },
  "quality_audit": {
    "general_sample_audi

In [ ]:
import shutil
from datetime import datetime, timezone

FINAL_DIR = os.path.join(SYNTHETIC_ROOT, "05_final_corpora")

rename_map = {
    "general_3k": ("general_3k_draft.csv", "general_3k_final.csv"),
    "general_6k": ("general_6k_draft.csv", "general_6k_final.csv"),
    "general_12k": ("general_12k_draft.csv", "general_12k_final.csv"),
    "banking_3k": ("banking_3k_draft.csv", "banking_3k_final.csv"),
}

manifest["files"] = {}
for name, (old_name, new_name) in rename_map.items():
    old_path = os.path.join(FINAL_DIR, old_name)
    new_path = os.path.join(FINAL_DIR, new_name)
    shutil.copyfile(old_path, new_path)
    with open(new_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        row_count = sum(1 for _ in reader)
    manifest["files"][name] = {
        "filename": new_name,
        "sha256": file_sha256(new_path),
        "row_count": row_count,
    }
    print(f"Copied {old_name} -> {new_name} ({row_count} rows)")

manifest["frozen_at_utc"] = datetime.now(timezone.utc).isoformat()

MANIFEST_PATH = os.path.join(SYNTHETIC_ROOT, "06_final_audit", "corpus_manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print()
print("Manifest updated with final filenames:", MANIFEST_PATH)

Copied general_3k_draft.csv -> general_3k_final.csv (3000 rows)
Copied general_6k_draft.csv -> general_6k_final.csv (6000 rows)
Copied general_12k_draft.csv -> general_12k_final.csv (12000 rows)
Copied banking_3k_draft.csv -> banking_3k_final.csv (3000 rows)

Manifest updated with final filenames: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/03_synthetic_data/06_final_audit/corpus_manifest.json
